In [ ]:
!pip install pandas
!pip install pycountry
!pip install plotly
!pip install -U kaleido


In [ ]:
# ============================================================
# 🎨 DESIGN REFACTOR PARA GRÁFICO CIRCULAR – CORRIGIDO
# ============================================================
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.lines import Line2D  # necessário para legenda customizada

# ------------------------------------------------------------
# 1. Títulos dos megatópicos
# ------------------------------------------------------------
megatopicos_titles = {
    1: "M1 – Intimate partner and domestic violence",
    2: "M2 – Children, adolescents and youth",
    3: "M3 – Minority Groups and Vulnerable Populations",
    4: "M4 – Humanitarian, migration and armed conflicts",
    5: "M5 – Health and social support",
    6: "M6 – Locations and contextual factors",
    7: "M7 – Lethal and Sexual Violence"
}

# ------------------------------------------------------------
# 2. Lista de tópicos por megatópico
# ------------------------------------------------------------
megatopicos_dict = {
    1: ["T1 – IPV in Multiple Contexts", "T4 – Domestic Violence & Culture",
        "T11 – Pregnancy & Childbirth", "T26 – Norms & Attitudes Toward IPV",
        "T30 – IPV Prevalence & Determinants"],
    2: ["T2 – Adolescent & Youth Violence", "T15 – Educational Settings",
        "T23 – Dating Violence & Youth Victimization", "T29 – Childhood Maltreatment"],
    3: ["T3 – Sex Workers & Trafficking", "T10 – People with Disabilities",
        "T12 – Elder Abuse & Neglect", "T13 – Drug Use & Urban Marginalization",
        "T16 – Sexual, Gender & Ethnic Minorities", "T18 – Economic Interventions & Empowerment"],
    4: ["T17 – Migrants & Asylum Seekers", "T24 – War/Armed Conflict",
        "T25 – Humanitarian Services"],
    5: ["T5 – COVID-19", "T7 – Survivor Experiences & Community Response",
        "T8 – HIV Contexts", "T31 – Trauma & Mental Health",
        "T32 – Reproductive Health, Abortion & Contraception"],
    6: ["T6 – Workplace Violence", "T9 – Emergency Services & Police Contexts",
        "T19 – Sexual Harassment & Institutional", "T21 – Global & Structural Patterns",
        "T27 – Community & Neighborhood Context", "T28 – Spatial Distribution of Lethal Violence"],
    7: ["T14 – GBV & Female Mortality in Brazil",
        "T20 – IPV Homicides", "T22 – IPV Sexual Violence"]
}

# ------------------------------------------------------------
# 3. Carregar dados
# ------------------------------------------------------------
df = pd.read_csv("/content/drive/MyDrive/geo_2/localizacoes_processadas_melhorado_v2.csv")

# Substituição de possíveis valores inconsistentes
df["Megatopico"] = df["Megatopico"].replace({11: 1})

# ------------------------------------------------------------
# 4. Preparar contagem por Megatópico e Tópico
# ------------------------------------------------------------
contagem = (
    df.groupby(["Megatopico", "Topico"])
      .size()
      .reset_index(name="n")
)

# Criação de labels curtos para os tópicos
contagem["label_curto"] = contagem["Topico"].apply(lambda x: f"T{int(x)}")
contagem = contagem.sort_values(by=["Megatopico", "n"], ascending=[True, False])

# ------------------------------------------------------------
# 5. Paleta de cores e fonte
# ------------------------------------------------------------
plt.rcParams['font.family'] = 'serif'
meg_colors = ["#1F3A70", "#E85C91", "#3C9D9B", "#F1B24A", "#8C6AA0", "#FF7F50", "#7F7F7F"]
cor_por_megatopico = {mt: meg_colors[(mt-1) % len(meg_colors)] for mt in sorted(contagem["Megatopico"].unique())}

# ------------------------------------------------------------
# 6. Cálculo dos ângulos e alturas
# ------------------------------------------------------------
total_topics = len(contagem)
angle_per_topic = 2 * np.pi / total_topics
max_n = contagem["n"].max()
min_n = contagem["n"].min()

angles, heights, colors_list, labels, megatopics_for_labels = [], [], [], [], []

for i, (_, row) in enumerate(contagem.iterrows()):
    angle = i * angle_per_topic + angle_per_topic/2
    height = 0.3 + 0.7 * (row["n"] - min_n) / (max_n - min_n)
    angles.append(angle)
    heights.append(height)
    colors_list.append(cor_por_megatopico[row["Megatopico"]])
    labels.append(row["label_curto"])
    megatopics_for_labels.append(row["Megatopico"])

# ------------------------------------------------------------
# 7. Plot circular
# ------------------------------------------------------------
fig, ax = plt.subplots(figsize=(15,15), subplot_kw=dict(polar=True))
fig.patch.set_facecolor('white')

# Outer ring
ax.bar(
    angles, heights, width=angle_per_topic*0.85, bottom=0.6,
    color=colors_list, edgecolor='white', linewidth=1.2, alpha=0.9
)

# Círculos concêntricos
valores_referencia = [10, 15, 20, 25, 30]
def n_to_height(n):
    return 0.3 + 0.7 * (n - min_n) / (max_n - min_n)

for v in valores_referencia:
    if v < min_n or v > max_n:
        continue
    r = 0.6 + n_to_height(v)
    ax.plot(np.linspace(0, 2*np.pi, 500), [r]*500, color="#B0B0B0", linewidth=0.8, alpha=0.7, linestyle="--")
    ax.text(np.pi/2, r, f"{v}", ha='center', va='bottom', fontsize=14, color="#444444")

# Inner ring (Megatópicos)
mega_data = {}
lista_megatopicos = sorted(contagem["Megatopico"].unique())

for mt in lista_megatopicos:
    subset = [i for i, m in enumerate(megatopics_for_labels) if m == mt]
    start_idx, end_idx = subset[0], subset[-1]
    mega_angle = (angles[start_idx] + angles[end_idx]) / 2
    mega_width = (end_idx - start_idx + 1) * angle_per_topic * 0.9
    mega_data[mt] = (mega_angle, mega_width)

for mt, (mega_angle, mega_width) in mega_data.items():
    ax.bar(mega_angle, 0.5, width=mega_width, bottom=0.1,
           color=cor_por_megatopico[mt], edgecolor='white', linewidth=2, alpha=0.85)

# Labels dos tópicos
for angle, label, height in zip(angles, labels, heights):
    ax.text(angle, 0.6 + height + 0.05, label, ha='center', va='center', fontsize=16, fontweight='bold')

# Labels dos megatópicos
for mt, (mega_angle, _) in mega_data.items():
    ax.text(mega_angle, 0.35, f"M{mt}", ha='center', va='center', fontsize=24, fontweight='bold', color='white')

# ------------------------------------------------------------
# 8. Legenda
# ------------------------------------------------------------
legend_elements = []
marker_map = {str(i): "o" for i in range(1, 40)}

for mt, topics in megatopicos_dict.items():
    # título do megatópico
    legend_elements.append(Line2D([], [], color="white", linestyle='none', label=f"{megatopicos_titles[mt]}"))
    # tópicos
    for topic_full in topics:
        tnum = topic_full.split("–")[0].strip()[1:]
        marker = marker_map.get(tnum, "o")
        legend_elements.append(Line2D([], [], marker=marker, color=cor_por_megatopico[mt],
                                      markersize=10, linestyle="none", markeredgecolor="white",
                                      markeredgewidth=0.8, label=f"   {topic_full}"))

ax.legend(handles=legend_elements, title="Megatopics & Topics",
          fontsize=13, title_fontsize=18, bbox_to_anchor=(1.12, 1), loc="upper left", frameon=True)

# Finalização
ax.set_ylim(0, 1.5)
ax.axis('off')
plt.subplots_adjust(top=0.95)
plt.savefig("/content/grafico_megatopicos_refinado.png", dpi=300, bbox_inches="tight", facecolor='white')
plt.show()

print("✅ Gráfico salvo como: /content/grafico_megatopicos_refinado.png")

# ------------------------------------------------------------
# Contagem de tópicos com títulos e porcentagens
# ------------------------------------------------------------
topicos_dict = {
    1: "IPV in Multiple Contexts", 4: "Domestic Violence & Culture", 11: "Pregnancy & Childbirth",
    26: "Norms & Attitudes Toward IPV", 30: "IPV Prevalence & Determinants",
    2: "Adolescent & Youth Violence", 15: "Educational Settings", 23: "Dating Violence & Youth Victimization",
    29: "Childhood Maltreatment", 3: "Sex Workers & Trafficking", 10: "People with Disabilities",
    12: "Elder Abuse & Neglect", 13: "Drug Use & Urban Marginalization", 16: "Sexual, Gender & Ethnic Minorities",
    18: "Economic Interventions & Empowerment", 17: "Migrants & Asylum Seekers", 24: "War / Armed Conflict",
    25: "Humanitarian Services", 5: "COVID-19", 7: "Survivor Experiences & Community Response",
    8: "HIV Contexts", 31: "Trauma & Mental Health", 32: "Reproductive Health, Abortion & Contraception",
    6: "Workplace Violence", 9: "Emergency Services & Police Contexts", 19: "Sexual Harassment & Institutional",
    21: "Global & Structural Patterns", 27: "Community & Neighborhood Context", 28: "Spatial Distribution of Lethal Violence",
    14: "GBV & Female Mortality in Brazil", 20: "IPV Homicides", 22: "IPV Sexual Violence"
}

contagem_topicos = df['Topico'].value_counts().reset_index()
contagem_topicos.columns = ['Topico', 'Count']
contagem_topicos['Titulo'] = contagem_topicos['Topico'].map(topicos_dict)
contagem_topicos['Perc'] = (contagem_topicos['Count'] / contagem_topicos['Count'].sum() * 100).round(2)
print("\nContagem por TÓPICO (com títulos e porcentagens):")
print(contagem_topicos)

# Contagem por Megatópico
def map_megatopico(topico):
    for key, lista in megatopicos_dict.items():
        if topico in lista:
            return megatopicos_titles[key]
    return "Sem classificação"

df['Megatopico'] = df['Topico'].apply(map_megatopico)



contagem_megatopicos = df['Megatopico'].value_counts().reset_index()
contagem_megatopicos.columns = ['Megatopico', 'Count']
contagem_megatopicos['Perc'] = (contagem_megatopicos['Count'] / contagem_megatopicos['Count'].sum() * 100).round(2)
print("\nContagem por MEGATÓPICO (com porcentagens):")
print(contagem_megatopicos)

# ============================================================
# SEGUNDO PRINT — CONTAGEM ORGANIZADA POR MEGATÓPICOS
# ============================================================

# Mapeamento auxiliar: T1, T2... para Megatópico
topico_to_mega = {}
for meg, lista in megatopicos_dict.items():
    for t in lista:
        cod = int(t.split("–")[0].replace("T", "").strip())
        topico_to_mega[cod] = meg

df["MegaNum"] = df["Topico"].map(topico_to_mega)

# Garantir ordenação correta
df["MegaNum"] = df["MegaNum"].fillna(-1)

# Contagem por tópico dentro de cada megatópico
contagem_detalhada = (
    df.groupby(["MegaNum", "Topico"])
      .size()
      .reset_index(name="Count")
      .sort_values(["MegaNum", "Count"], ascending=[True, False])
)

# Adicionar títulos
contagem_detalhada["Megatopico_Titulo"] = contagem_detalhada["MegaNum"].map(megatopicos_titles)
contagem_detalhada["Topico_Titulo"] = contagem_detalhada["Topico"].map(topicos_dict)
contagem_detalhada["Label"] = contagem_detalhada["Topico"].apply(lambda x: f"T{int(x)}")

# Contagem total por megatópico
total_por_mega = (
    contagem_detalhada.groupby("MegaNum")["Count"].sum()
)

# Adicionar porcentagem
contagem_detalhada["Perc_no_Megatopico"] = contagem_detalhada.apply(
    lambda row: (row["Count"] / total_por_mega[row["MegaNum"]] * 100).round(2),
    axis=1
)

# Impressão organizada
print("\n====================================================")
print("📌 CONTAGEM ORGANIZADA POR MEGATÓPICOS")
print("====================================================\n")

for mega in sorted(contagem_detalhada["MegaNum"].unique()):
    if mega == -1:
        continue
    titulo = megatopicos_titles[mega]
    total = total_por_mega[mega]

    print(f"\n🔷 MEGATÓPICO M{mega}: {titulo}")
    print(f"   Total de entradas: {total}\n")

    subset = contagem_detalhada[contagem_detalhada["MegaNum"] == mega]

    for _, row in subset.iterrows():
        print(f"   {row['Label']}: {row['Topico_Titulo']} — {row['Count']} ocorrências "
              f"({row['Perc_no_Megatopico']}%)")

    print("\n" + "-"*55)

# Tabela final compacta
print("\n====================================================")
print("📊 TABELA FINAL (MEGATÓPICOS > TÓPICOS)")
print("====================================================")
print(contagem_detalhada[[
    "MegaNum", "Megatopico_Titulo", "Label", "Topico_Titulo", "Count", "Perc_no_Megatopico"
]].reset_index(drop=True))



In [ ]:
import pandas as pd

# Caminho do arquivo CSV
csv_path = "/content/drive/MyDrive/geo_2/matched_articles_info_com_texto_corrigido.csv"

# Carregar o DataFrame
df = pd.read_csv(csv_path)

print(f"✅ DataFrame carregado de: {csv_path}")
print("\nContagem de entradas em branco (NaN, null, strings vazias ou apenas espaços) por coluna:")

# Iterar por cada coluna e contar entradas em branco
blank_counts = {}
for col in df.columns:
    # Contar NaN/null
    nan_count = df[col].isnull().sum()
    # Contar strings vazias ou apenas espaços (apenas para colunas de string)
    if df[col].dtype == 'object':
        empty_string_count = df[col].astype(str).apply(lambda x: x.strip() == '').sum()
    else:
        empty_string_count = 0

    total_blank = nan_count + empty_string_count
    if total_blank > 0:
        blank_counts[col] = total_blank

# Exibir os resultados
if blank_counts:
    for col, count in blank_counts.items():
        print(f"- {col}: {count}")
else:
    print("Não foram encontradas entradas em branco em nenhuma coluna.")

print(f"\nTotal de entradas no DataFrame: {len(df)}")

In [ ]:
# ================================================
# 📌 t-SNE COM MEGATÓPICOS (TÍTULOS + TÓPICOS)
# ================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.manifold import TSNE
import matplotlib.colors as mcolors

# ----------------------------------------------
# 0. Títulos dos Megatópicos + Dicionário Revisado
# ----------------------------------------------
megatopicos_titles = {
    1: "M1 – Intimate partner and domestic violence",
    2: "M2 – Children, adolescents and youth",
    3: "M3 – Minority Groups and Vulnerable Populations",
    4: "M4 – Humanitarian, migration and armed conflicts",
    5: "M5 – Health and social support",
    6: "M6 – Locations and contextual factors",
    7: "M7 – Lethal and Sexual Violence"
}

megatopicos_dict = {
    1: ["T1 – IPV in Multiple Contexts", "T4 – Domestic Violence & Culture",
        "T11 – Pregnancy & Childbirth", "T26 – Norms & Attitudes Toward IPV",
        "T30 – IPV Prevalence & Determinants"],

    2: ["T2 – Adolescent & Youth Violence", "T15 – Educational Settings",
        "T23 – Dating Violence & Youth Victimization", "T29 – Childhood Maltreatment"],

    3: ["T3 – Sex Workers & Trafficking", "T10 – People with Disabilities",
        "T12 – Elder Abuse & Neglect", "T13 – Drug Use & Urban Marginalization",
        "T16 – Sexual, Gender & Ethnic Minorities", "T18 – Economic Interventions & Empowerment"],

    4: ["T17 – Migrants & Asylum Seekers", "T24 – War/Armed Conflict",
        "T25 – Humanitarian Services"],

    5: ["T5 – COVID-19", "T7 – Survivor Experiences & Community Response",
        "T8 – HIV Contexts", "T31 – Trauma & Mental Health",
        "T32 – Reproductive Health, Abortion & Contraception"],

    6: ["T6 – Workplace Violence", "T9 – Emergency Services & Police Contexts",
        "T19 – Sexual Harassment & Institutional", "T21 – Global & Structural Patterns",
        "T27 – Community & Neighborhood Context", "T28 – Spatial Distribution of Lethal Violence"],

    7: ["T14 – GBV & Female Mortality in Brazil",
        "T20 – IPV Homicides", "T22 – IPV Sexual Violence"]
}

# ---------------------------------------------
# 1. Marcadores (símbolos) para distribuir entre tópicos
# ---------------------------------------------
topic_markers = ["o", "s", "D", "^", "v", "P", "X", "<", ">", "*", "h"]

# ---------------------------------------------
# 🎨 2. DESIGN TOKENS (mantidos do teu modelo)
# ---------------------------------------------
plt.rcParams["font.family"] = "serif"

MEGA_COLORS = [
    "#1F3A70", "#E85C91", "#3C9D9B",
    "#F1B24A", "#8C6AA0", "#FF7F50", "#7F7F7F"
]

cor_por_megatopico = {
    mt: MEGA_COLORS[(mt-1) % len(MEGA_COLORS)]
    for mt in megatopicos_dict
}

SCATTER_ALPHA = 0.85
SCATTER_SIZE = 95
EDGE_COLOR = "white"
EDGE_WIDTH = 0.9
TITLE_COLOR = "#1F3A70"

# ---------------------------------------------
# 3. Carregar CSV
# ---------------------------------------------
df = pd.read_csv("/content/drive/MyDrive/geo_2/final_geo2.csv")
df = df[~df["cleaned"].isna() & (df["cleaned"].astype(str).str.strip() != "")]
df = df.reset_index(drop=True)

# ---------------------------------------------
# 4. TF-IDF
# ---------------------------------------------
vectorizer = TfidfVectorizer(
    stop_words="english",
    max_df=0.9,
    min_df=5,
    ngram_range=(1, 2),
    max_features=5000
)

X = vectorizer.fit_transform(df["cleaned"])

# ---------------------------------------------
# 5. t-SNE
# ---------------------------------------------
tsne = TSNE(
    n_components=2,
    random_state=42,
    perplexity=30,
    init='pca',
    learning_rate='auto',
    max_iter=2000
)
X_embedded = tsne.fit_transform(X.toarray())
df["tsne_x"] = X_embedded[:, 0]
df["tsne_y"] = X_embedded[:, 1]

# ---------------------------------------------
# 6. Construção de mapa de marcadores por tópico
# ---------------------------------------------
marker_map = {}
for mt, topics in megatopicos_dict.items():
    available = topic_markers[:len(topics)]
    for topic_name, mk in zip(topics, available):

        # extrai o número do tópico (T26 → 26)
        t_number = topic_name.split("–")[0].strip()[1:]
        marker_map[t_number] = mk

# ---------------------------------------------
# 7. Plot
# ---------------------------------------------
fig, ax = plt.subplots(figsize=(20, 13))
fig.patch.set_facecolor("white")

for mt, topics in megatopicos_dict.items():
    subset = df[df["Megatopico"] == mt]

    for topic_full in topics:

        tnum = topic_full.split("–")[0].strip()[1:]
        subsub = subset[subset["Topico"] == int(tnum)]
        if subsub.empty:
            continue

        ax.scatter(
            subsub["tsne_x"],
            subsub["tsne_y"],
            c=[cor_por_megatopico[mt]] * len(subsub),
            marker=marker_map[tnum],
            s=SCATTER_SIZE,
            alpha=SCATTER_ALPHA,
            edgecolors=EDGE_COLOR,
            linewidth=EDGE_WIDTH,
            label=f"{megatopicos_titles[mt]}\n   {topic_full}"
        )

# ---------------------------------------------
# 8. Legenda — Megatópico 1x + Tópicos indentados
# ---------------------------------------------
from matplotlib.lines import Line2D

legend_elements = []

for mt, topics in megatopicos_dict.items():

    # --- título do megatópico (aparece UMA VEZ) ---
    legend_elements.append(
        Line2D([], [], color="white", marker=None, linestyle='none',
               label=f"{megatopicos_titles[mt]}")
    )

    # --- tópicos abaixo do título ---
    for topic_full in topics:
        tnum = topic_full.split("–")[0].strip()[1:]   # extrai número
        marker = marker_map[tnum]

        legend_elements.append(
            Line2D([], [],
                   marker=marker,
                   color=cor_por_megatopico[mt],
                   markersize=10,
                   linestyle="none",
                   markeredgecolor="white",
                   markeredgewidth=0.8,
                   alpha=0.9,
                   label=f"   {topic_full}")
        )

# aplicar legenda
ax.legend(
    handles=legend_elements,
    title="Megatopics & Topics",
    fontsize=13,
    title_fontsize=18,
    bbox_to_anchor=(1.05, 1),
    loc="upper left",
    frameon=True
)

# ---------------------------------------------
# 9. Títulos e eixos (com o design solicitado)
# ---------------------------------------------

ax.set_xlabel("t-SNE 1", fontsize=14, fontweight='bold')
ax.set_ylabel("t-SNE 2", fontsize=14, fontweight='bold')
ax.grid(True, linestyle="--", alpha=0.28)

plt.tight_layout()
plt.show()


In [ ]:
# ================================================
# 📌 UMAP COM MEGATÓPICOS (TÍTULOS + TÓPICOS)
# ================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
import umap
import matplotlib.colors as mcolors

# ----------------------------------------------
# 0. Títulos dos Megatópicos + Dicionário Revisado
# ----------------------------------------------
megatopicos_titles = {
    1: "M1 – Intimate partner and domestic violence",
    2: "M2 – Children, adolescents and youth",
    3: "M3 – Minority Groups and Vulnerable Populations",
    4: "M4 – Humanitarian, migration and armed conflicts",
    5: "M5 – Health and social support",
    6: "M6 – Locations and contextual factors",
    7: "M7 – Lethal and Sexual Violence"
}

megatopicos_dict = {
    1: ["T1 – IPV in Multiple Contexts", "T4 – Domestic Violence & Culture",
        "T11 – Pregnancy & Childbirth", "T26 – Norms & Attitudes Toward IPV",
        "T30 – IPV Prevalence & Determinants"],

    2: ["T2 – Adolescent & Youth Violence", "T15 – Educational Settings",
        "T23 – Dating Violence & Youth Victimization", "T29 – Childhood Maltreatment"],

    3: ["T3 – Sex Workers & Trafficking", "T10 – People with Disabilities",
        "T12 – Elder Abuse & Neglect", "T13 – Drug Use & Urban Marginalization",
        "T16 – Sexual, Gender & Ethnic Minorities", "T18 – Economic Interventions & Empowerment"],

    4: ["T17 – Migrants & Asylum Seekers", "T24 – War/Armed Conflict",
        "T25 – Humanitarian Services"],

    5: ["T5 – COVID-19", "T7 – Survivor Experiences & Community Response",
        "T8 – HIV Contexts", "T31 – Trauma & Mental Health",
        "T32 – Reproductive Health, Abortion & Contraception"],

    6: ["T6 – Workplace Violence", "T9 – Emergency Services & Police Contexts",
        "T19 – Sexual Harassment & Institutional", "T21 – Global & Structural Patterns",
        "T27 – Community & Neighborhood Context", "T28 – Spatial Distribution of Lethal Violence"],

    7: ["T14 – GBV & Female Mortality in Brazil",
        "T20 – IPV Homicides", "T22 – IPV Sexual Violence"]
}

# ---------------------------------------------
# 1. Marcadores (símbolos)
# ---------------------------------------------
topic_markers = ["o", "s", "D", "^", "v", "P", "X", "<", ">", "*", "h"]

# ---------------------------------------------
# 🎨 2. DESIGN TOKENS
# ---------------------------------------------
plt.rcParams["font.family"] = "serif"

MEGA_COLORS = [
    "#1F3A70", "#E85C91", "#3C9D9B",
    "#F1B24A", "#8C6AA0", "#FF7F50", "#7F7F7F"
]

cor_por_megatopico = {
    mt: MEGA_COLORS[(mt-1) % len(MEGA_COLORS)]
    for mt in megatopicos_dict
}

SCATTER_ALPHA = 0.85
SCATTER_SIZE = 95
EDGE_COLOR = "white"
EDGE_WIDTH = 0.9
TITLE_COLOR = "#1F3A70"

# ---------------------------------------------
# 3. Carregar CSV
# ---------------------------------------------
df = pd.read_csv("/content/drive/MyDrive/geo_2/final_geo2.csv")
df = df[~df["cleaned"].isna() & (df["cleaned"].astype(str).str.strip() != "")]
df = df.reset_index(drop=True)

# ---------------------------------------------
# 4. TF-IDF
# ---------------------------------------------
vectorizer = TfidfVectorizer(
    stop_words="english",
    max_df=0.9,
    min_df=5,
    ngram_range=(1, 2),
    max_features=5000
)

X = vectorizer.fit_transform(df["cleaned"])

# ---------------------------------------------
# 5. UMAP (versão estável e interpretável)
# ---------------------------------------------
umap_model = umap.UMAP(
    n_neighbors=25,
    min_dist=0.15,
    n_components=2,
    metric="cosine",
    random_state=42
)

X_umap = umap_model.fit_transform(X)
df["umap_x"] = X_umap[:, 0]
df["umap_y"] = X_umap[:, 1]

# ---------------------------------------------
# 6. Construção de mapa de marcadores por tópico
# ---------------------------------------------
marker_map = {}

for mt, topics in megatopicos_dict.items():
    available = topic_markers[:len(topics)]
    for topic_name, mk in zip(topics, available):

        t_number = topic_name.split("–")[0].strip()[1:]  # extrai número
        marker_map[t_number] = mk

# ---------------------------------------------
# 7. Plot
# ---------------------------------------------
fig, ax = plt.subplots(figsize=(20, 13))
fig.patch.set_facecolor("white")

for mt, topics in megatopicos_dict.items():
    subset = df[df["Megatopico"] == mt]

    for topic_full in topics:

        tnum = topic_full.split("–")[0].strip()[1:]
        subsub = subset[subset["Topico"] == int(tnum)]
        if subsub.empty:
            continue

        ax.scatter(
            subsub["umap_x"],
            subsub["umap_y"],
            c=[cor_por_megatopico[mt]] * len(subsub),
            marker=marker_map[tnum],
            s=SCATTER_SIZE,
            alpha=SCATTER_ALPHA,
            edgecolors=EDGE_COLOR,
            linewidth=EDGE_WIDTH,
            label=f"{megatopicos_titles[mt]}\n   {topic_full}"
        )

# ---------------------------------------------
# 8. Legenda — Megatópico 1x + Tópicos indentados
# ---------------------------------------------
from matplotlib.lines import Line2D

legend_elements = []

for mt, topics in megatopicos_dict.items():

    legend_elements.append(
        Line2D([], [], color="white", marker=None, linestyle='none',
               label=f"{megatopicos_titles[mt]}")
    )

    for topic_full in topics:
        tnum = topic_full.split("–")[0].strip()[1:]
        marker = marker_map[tnum]

        legend_elements.append(
            Line2D([], [],
                   marker=marker,
                   color=cor_por_megatopico[mt],
                   markersize=10,
                   linestyle="none",
                   markeredgecolor="white",
                   markeredgewidth=0.8,
                   alpha=0.9,
                   label=f"   {topic_full}")
        )

ax.legend(
    handles=legend_elements,
    title="Megatopics & Topics",
    fontsize=13,
    title_fontsize=18,
    bbox_to_anchor=(1.05, 1),
    loc="upper left",
    frameon=True
)

# ---------------------------------------------
# 9. Títulos e eixos
# ---------------------------------------------
ax.set_xlabel("UMAP 1", fontsize=14, fontweight='bold')
ax.set_ylabel("UMAP 2", fontsize=14, fontweight='bold')
ax.grid(True, linestyle="--", alpha=0.28)

plt.tight_layout()
plt.show()


In [ ]:
# ============================================================================
# 1. Importações
# ============================================================================
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity


# ============================================================================
# 3. Carregar CSV
# ============================================================================
df = pd.read_csv("/content/drive/MyDrive/geo_2/final_geo2.csv")
df = df[~df["cleaned"].isna() & (df["cleaned"].astype(str).str.strip() != "")]
df = df.reset_index(drop=True)


# ============================================================================
# 4. Vetorização TF-IDF
# ============================================================================
vectorizer = TfidfVectorizer(
    stop_words="english",
    max_df=0.9,
    min_df=5,
    ngram_range=(1, 2),
    max_features=5000
)

X = vectorizer.fit_transform(df["cleaned"])


# ============================================================================
# 5. Criar centroides TF-IDF por tópico
# ============================================================================
mega_order = [
    1, 4, 11, 26, 30,
    2, 15, 23, 29,
    3, 10, 12, 13, 16, 18,
    17, 24, 25,
    5, 7, 8, 31, 32,
    6, 9, 19, 21, 27, 28,
    14, 20, 22
]

topics = [t for t in mega_order if t in df["Topico"].unique()]

topic_centroids = {}

for t in topics:
    mask = (df["Topico"] == t).values
    subset = X[mask]

    if subset.shape[0] == 0:
        continue

    centroid = subset.mean(axis=0)

    # 🔥 CONVERSÃO NECESSÁRIA: evita erro np.matrix
    centroid = np.asarray(centroid).ravel()

    topic_centroids[t] = centroid


# ============================================================================
# 6. Construir matriz com os centroides
# ============================================================================
centroid_matrix = np.vstack([topic_centroids[t] for t in topics])

# Similaridade coseno
similarity_matrix = cosine_similarity(centroid_matrix)

# Threshold para realçar alta similaridade
threshold = 0.80
similarity_thresh = np.where(similarity_matrix >= threshold, similarity_matrix, np.nan)


# ============================================================================
# 7. Heatmap Completo — Estilo Refinado
# ============================================================================
plt.figure(figsize=(16, 10))
ax = sns.heatmap(
    similarity_matrix,
    cmap="YlGnBu",
    linewidths=1,
    linecolor="#EAEAEA",
    cbar_kws={"label": "Cosine Similarity", "shrink": 0.8},
    annot=True,
    fmt=".2f",
    vmin=0,
    vmax=1,
    xticklabels=[f"T{t}" for t in topics],
    yticklabels=[f"T{t}" for t in topics]
)

ax.set_xticklabels(ax.get_xticklabels(), rotation=-35, ha="left", fontsize=11.5)
ax.set_yticklabels(ax.get_yticklabels(), fontsize=11.5, fontweight="semibold")
ax.set_xlabel("Topic", fontweight="bold", fontsize=12)
ax.set_ylabel("Topic", fontweight="bold", fontsize=12)

plt.tight_layout()
plt.show()

ax.set_ylabel("Topic", fontweight="bold", fontsize=12)

plt.tight_layout()
plt.show()


In [ ]:
# ============================================================
# 🎨 DESIGN REFACTOR — PLOT 1
# ============================================================
import pandas as pd
import matplotlib.pyplot as plt

# ----------------------------
# 1. Load data
# ----------------------------
df = pd.read_csv("/content/drive/MyDrive/geo_2/localizacoes_processadas_melhorado_v2.csv")

# Fix Megatopic 11 → 1
df["Megatopico"] = df["Megatopico"].replace({11: 1})

# Count per year
year_counts = df.groupby("year").size().reset_index(name="n")

# ----------------------------
# 2. Design tokens
# ----------------------------
plt.rcParams["font.family"] = "serif"

TITLE_COLOR = "#1F3A70"
LINE_COLOR = "#1F3A70"            # institutional color
MARKER_FACE = "#F1B24A"           # elegant contrast
MARKER_EDGE = "#1F3A70"

GRID_COLOR = "#7F7F7F"

# ----------------------------
# 3. Plot
# ----------------------------
fig, ax = plt.subplots(figsize=(14, 6))
fig.patch.set_facecolor("white")

ax.plot(
    year_counts["year"], year_counts["n"],
    marker="o",
    markersize=10,
    markerfacecolor=MARKER_FACE,
    markeredgecolor=MARKER_EDGE,
    markeredgewidth=1.5,
    linewidth=3,
    color=LINE_COLOR,
    alpha=0.9
)

# ----------------------------
# 4. Plot styling
# ----------------------------
ax.grid(
    axis="y",
    linestyle="--",
    linewidth=0.8,
    alpha=0.5,
    color=GRID_COLOR
)

ax.set_xlabel("Year", fontsize=14, fontweight="bold")
ax.set_ylabel("Total number of entries", fontsize=14, fontweight="bold")



# Margins and layout optimization
plt.tight_layout()
plt.show()

# ===============================================
# Publications count per year
# ===============================================

# 3. Count per year
year_counts_series = df["year"].value_counts().sort_index()

# 4. Percentage per year
year_percent = (year_counts_series / year_counts_series.sum() * 100).round(2)

# 5. Print
print("=== Number of publications per year ===")
print(year_counts_series)

print("\n=== Percentage of publications per year (%) ===")
print(year_percent)


In [ ]:
# === Gráfico 2: Distribuição percentual de megatópicos ao longo do tempo ===

import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.cm import get_cmap

# Carregar dados
df = pd.read_csv("/content/drive/MyDrive/geo_2/localizacoes_processadas_melhorado_v2.csv")

# Corrigir Megatópico 11 → 1
df["Megatopico"] = df["Megatopico"].replace({11: 1})

# Contagem por ano e megatópico
contagem = df.groupby(["year", "Megatopico"]).size().reset_index(name="n")

# Pivot para porcentagens
pivot = contagem.pivot(index="year", columns="Megatopico", values="n").fillna(0)

# Converter para proporções (%)
percentuais = pivot.div(pivot.sum(axis=1), axis=0) * 100

# Paleta suave Set2
palette = get_cmap("Set2")
colors = {mt: palette((mt-1) % 7) for mt in percentuais.columns}

# Plot
plt.figure(figsize=(12, 6))

for mt in percentuais.columns:
    plt.plot(
        percentuais.index,
        percentuais[mt],
        marker="o",
        linewidth=2,
        label=f"M{mt}",
        color=colors[mt]
    )

plt.grid(axis="y", linestyle="--", linewidth=0.5, alpha=0.6)
plt.xlabel("Ano", fontsize=11)
plt.ylabel("Percentual (%)", fontsize=11)
plt.title("Distribuição percentual dos megatópicos por ano", fontsize=14)
plt.legend(title="Megatópico", frameon=False)

plt.tight_layout()
plt.show()


In [ ]:
# ============================================================
# 🎨 DESIGN REFACTOR — 100% STACKED BAR CHART (corrigido)
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# ----------------------------
# 1. Design tokens
# ----------------------------
plt.rcParams["font.family"] = "serif"

TITLE_COLOR = "#1F3A70"
LABEL_COLOR = "#1F3A70"
GRID_COLOR = "#7F7F7F"

MEGA_COLORS = [
    "#1F3A70", "#E85C91", "#3C9D9B",
    "#F1B24A", "#8C6AA0", "#FF7F50", "#7F7F7F"
]

# ----------------------------
# 2. Títulos dos Megatópicos
# ----------------------------
megatopicos_titles = {
    1: "M1 – Intimate partner and domestic violence",
    2: "M2 – Children, adolescents and youth",
    3: "M3 – Minority Groups and Vulnerable Populations",
    4: "M4 – Humanitarian, migration and armed conflicts",
    5: "M5 – Health and social support",
    6: "M6 – Locations and contextual factors",
    7: "M7 – Lethal and Sexual Violence"
}

# ----------------------------
# 3. Carregar dataframe
# ----------------------------
df = pd.read_csv("/content/drive/MyDrive/geo_2/localizacoes_processadas_melhorado_v2.csv")

# Corrigir megatópico 11 -> 1
if "Megatopico" in df.columns:
    df["Megatopico"] = df["Megatopico"].replace({11: 1})

# ----------------------------
# 4. Garantir coluna year correta
# ----------------------------
# remover NaN, converter para int onde possível
if "year" not in df.columns:
    raise ValueError("A coluna 'year' não existe no CSV.")

years_raw = df["year"].dropna()
# tentar converter para inteiro (se já for inteiro, okay)
years_int = pd.to_numeric(years_raw, errors="coerce").dropna().astype(int)
# filtrar dataframe original para linhas com year válido
df = df.loc[df["year"].isin(years_int)]

# usar lista ordenada de anos realmente presentes
years = sorted(years_int.unique())
if len(years) == 0:
    raise ValueError("Não há anos válidos no dataset após limpeza.")

# ----------------------------
# 5. Agrupar dados por ano e megatópico
# ----------------------------
contagem = (
    df.groupby(["year", "Megatopico"])
      .size()
      .reset_index(name="n")
)

# Pivot com índice = year (apenas anos existentes)
pivot = contagem.pivot(index="year", columns="Megatopico", values="n").fillna(0).astype(int)

# Assegurar que as linhas estejam na ordem correta de 'years'
pivot = pivot.reindex(index=years, fill_value=0)

# Percentuais (por linha) — evitar divisão por zero
percentuais = pivot.div(pivot.sum(axis=1).replace(0, np.nan), axis=0).fillna(0) * 100

# ----------------------------
# 6. Print dos valores completos
# ----------------------------
print("\n=== Valores absolutos (número de artigos por ano e megatópico) ===")
print(pivot.to_string())
print("\n=== Valores percentuais (%) por ano e megatópico ===")
print(percentuais.round(2).to_string())

# ----------------------------
# 7. Plotagem (uso de posições X para evitar ticks extras)
# ----------------------------
fig, ax = plt.subplots(figsize=(16, 7))
fig.patch.set_facecolor("white")

x = np.arange(len(years))  # posições no eixo x
width = 0.9  # largura das barras (total, empilhadas)

bottom = np.zeros(len(years))

# Usar colunas ordenadas por número do megatópico se existirem
cols = sorted(percentuais.columns)

for mt in cols:
    vals = percentuais[mt].reindex(index=years, fill_value=0).values
    ax.bar(
        x,
        vals,
        bottom=bottom,
        color=MEGA_COLORS[(mt - 1) % len(MEGA_COLORS)],
        edgecolor="white",
        linewidth=1.0,
        alpha=0.9,
        label=megatopicos_titles.get(mt, f"M{mt}"),
        width=width
    )
    bottom += vals

# ----------------------------
# 8. Estilização refinada
# ----------------------------
# Ajustar rótulos do eixo x para mostrar os anos corretos (sem extras)
ax.set_xticks(x)
ax.set_xticklabels([str(y) for y in years], rotation=0, fontsize=11, color="#2C3E50")

ax.set_ylabel("Percentage (%)", fontsize=14, fontweight="bold", color=LABEL_COLOR)
ax.set_xlabel("Year", fontsize=14, fontweight="bold", color=LABEL_COLOR)

ax.tick_params(axis="y", labelsize=11, colors="#2C3E50")

ax.grid(axis="y", linestyle="--", linewidth=0.8, alpha=0.5, color=GRID_COLOR)

plt.legend(
    title="Megatopics",
    title_fontsize=14,
    fontsize=11,
    frameon=True,
    facecolor="white",
    edgecolor="#D0D8E8",
    loc="upper left",
    bbox_to_anchor=(1.02, 1)
)

plt.tight_layout()
plt.show()


In [ ]:
# 📊 Análise comparativa de países – versão final com filtragem ISO3 válida
# Autor: Felipe Correa Bitencourt
# Data: 2025-10-29

import pandas as pd
import pycountry
from plotly.subplots import make_subplots
import plotly.graph_objects as go
from typing import Optional, Set, Tuple
import logging

# ==========================================================
# Configuração de logging
# ==========================================================
logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")
logger = logging.getLogger(__name__)

# ==========================================================
# Classe principal para análise comparativa de países
# ==========================================================
class CountryAnalysisDashboard:
    """Classe para análise comparativa de países estudados vs países de origem."""

    def __init__(self, csv_path: str):
        self.csv_path = csv_path
        self.df = None
        self.comparativo_df = None
        self.pais_counts_iso = None
        self.pais_counts_loc = None

    # ==========================================================
    # 1️⃣ Carregar CSV
    # ==========================================================
    def load_data(self) -> None:
        """Carrega e prepara os dados do CSV processado."""
        try:
            self.df = pd.read_csv(self.csv_path)
            logger.info(f"✅ Dados carregados: {len(self.df)} registros no total.")
        except Exception as e:
            logger.error(f"Erro ao carregar arquivo CSV: {e}")
            raise

    # ==========================================================
    # 2️⃣ Processar coluna 'País' (estudos)
    # ==========================================================
    def process_studied_countries(self) -> pd.DataFrame:
        df_validos = self.df[self.df['País'].notna()].copy()
        df_validos = df_validos.assign(País=df_validos['País'].astype(str).str.replace(r'[,/]', ';'))
        df_exploded = df_validos.assign(País=df_validos['País'].str.split(';')).explode('País')
        df_exploded['País'] = df_exploded['País'].str.strip().str.upper()
        pais_counts = df_exploded['País'].value_counts().reset_index()
        pais_counts.columns = ['ISO3', 'Contagem_Estudo']
        self.pais_counts_iso = pais_counts
        logger.info(f"🌍 Países estudados processados: {len(pais_counts)} únicos.")
        return pais_counts

    # ==========================================================
    # 3️⃣ Processar países de origem (pais_location → ISO3)
    # ==========================================================
    def process_origin_countries(self) -> pd.DataFrame:
        df_validos = self.df[self.df['pais_location'].notna()].copy()
        df_validos['pais_location'] = df_validos['pais_location'].astype(str).str.replace(r'[,/]', ';')
        df_exploded = df_validos.assign(pais_location=df_validos['pais_location'].str.split(';')).explode('pais_location')
        df_exploded['pais_location'] = df_exploded['pais_location'].str.strip().str.upper()

        def iso2_para_iso3(code: str) -> Optional[str]:
            try:
                if len(code) == 2:
                    country = pycountry.countries.get(alpha_2=code)
                    return country.alpha_3 if country else None
                return code
            except Exception:
                return None

        df_exploded['ISO3'] = df_exploded['pais_location'].apply(iso2_para_iso3)
        df_exploded = df_exploded[df_exploded['ISO3'].notna()]
        pais_counts = df_exploded['ISO3'].value_counts().reset_index()
        pais_counts.columns = ['ISO3', 'Contagem_Origem']
        self.pais_counts_loc = pais_counts
        logger.info(f"🏫 Países de origem processados: {len(pais_counts)} únicos.")
        return pais_counts

    # ==========================================================
    # 4️⃣ Criar dataframe comparativo
    # ==========================================================
    def create_comparison(self) -> pd.DataFrame:
        if self.pais_counts_iso is None or self.pais_counts_loc is None:
            raise ValueError("Os dataframes de países estudados e de origem precisam ser processados antes.")

        def is_valid_iso3(code):
            try:
                return pycountry.countries.get(alpha_3=code) is not None
            except:
                return False

        df_estudo_validos = self.pais_counts_iso[self.pais_counts_iso['ISO3'].apply(is_valid_iso3)]
        df_origem_validos = self.pais_counts_loc[self.pais_counts_loc['ISO3'].apply(is_valid_iso3)]

        df_merged = pd.merge(
            df_estudo_validos,
            df_origem_validos,
            on='ISO3',
            how='outer'
        ).fillna(0)

        df_merged['Total'] = df_merged['Contagem_Estudo'] + df_merged['Contagem_Origem']
        df_merged['Diferença'] = df_merged['Contagem_Estudo'] - df_merged['Contagem_Origem']

        self.comparativo_df = df_merged
        logger.info(f"📊 DataFrame comparativo criado com {len(df_merged)} países válidos.")
        return df_merged

    # ==========================================================
    # 5️⃣ Funções auxiliares e resumo
    # ==========================================================
    def get_country_sets(self) -> Tuple[Set, Set, Set]:
        def is_valid_iso3(code):
            try:
                return pycountry.countries.get(alpha_3=code) is not None
            except:
                return False

        set_estudo = set(self.pais_counts_iso[self.pais_counts_iso['ISO3'].apply(is_valid_iso3)]['ISO3'])
        set_origem = set(self.pais_counts_loc[self.pais_counts_loc['ISO3'].apply(is_valid_iso3)]['ISO3'])
        intersecao = set_estudo & set_origem
        return set_estudo, set_origem, intersecao

    def print_summary(self) -> None:
        set_estudo, set_origem, intersecao = self.get_country_sets()
        print("\n" + "="*60)
        print("📊 RESUMO DA ANÁLISE DE PAÍSES")
        print("="*60)
        print(f"🌍 Países estudados: {len(set_estudo)}")
        print(f"🏫 Países de origem: {len(set_origem)}")
        print(f"🔁 Presentes em ambas as colunas: {len(intersecao)}")
        print(f"📖 Apenas estudados: {len(set_estudo - set_origem)}")
        print(f"✍️ Apenas de origem: {len(set_origem - set_estudo)}")
        print("="*60)

    # ==========================================================
    # 6️⃣ Criação dos mapas e dashboard
    # ==========================================================
    def create_choropleth_map(self, data: pd.DataFrame, z_column: str,
                              colorscale: str, title: str, zmid: int = None,
                              reversescale=False, max_limit: float = None) -> go.Choropleth:
        custom_colorscale = [
            [0.0, "#08306B"],
            [0.3, "#2171B5"],
            [0.5, "#FFFFFF"],
            [1.0, "#FF0000"]
        ]
        map_config = {
            'locations': data['ISO3'],
            'z': data[z_column],
            'colorscale': custom_colorscale if zmid == 0 else colorscale,
            'colorbar_title': title,
            'zmid': zmid if zmid is not None else 0,
        }
        max_abs = max(abs(data[z_column].min()), abs(data[z_column].max()))
        if max_limit is not None:
            max_abs = min(max_abs, max_limit)
        if zmid == 0:
            map_config['zmin'] = -max_abs
            map_config['zmax'] = max_abs
        else:
            map_config['zmin'] = data[z_column].min()
            map_config['zmax'] = data[z_column].max()
        return go.Choropleth(**map_config)

    def create_dashboard(self) -> go.Figure:
        fig = make_subplots(
            rows=2, cols=2,
            subplot_titles=(
                "Países estudados",
                "Países de origem",
                "Diferença (Estudo - Origem)",
                "Dispersão: Estudo × Origem"
            ),
            specs=[
                [{"type": "choropleth"}, {"type": "choropleth"}],
                [{"type": "choropleth"}, {"type": "xy"}]
            ],
            vertical_spacing=0.08,
            horizontal_spacing=0.05
        )
        fig.add_trace(self.create_choropleth_map(self.pais_counts_iso, 'Contagem_Estudo', "YlOrRd", "Estudo"), row=1, col=1)
        fig.add_trace(self.create_choropleth_map(self.pais_counts_loc, 'Contagem_Origem', "YlOrRd", "Origem"), row=1, col=2)
        fig.add_trace(self.create_choropleth_map(self.comparativo_df, 'Diferença', "YlOrRd", "Diferença", zmid=0), row=2, col=1)

        scatter = go.Scatter(
            x=self.comparativo_df['Contagem_Estudo'],
            y=self.comparativo_df['Contagem_Origem'],
            mode='markers+text',
            text=self.comparativo_df['ISO3'],
            textposition="top center",
            marker=dict(size=10, color=self.comparativo_df['Total'], colorscale='Viridis', showscale=True, opacity=0.7),
            hovertemplate="<b>%{text}</b><br>Estudo: %{x}<br>Origem: %{y}<extra></extra>",
            name="Dispersão"
        )
        fig.add_trace(scatter, row=2, col=2)

        max_val = max(self.comparativo_df[['Contagem_Estudo', 'Contagem_Origem']].max())
        fig.add_shape(type="line", x0=0, y0=0, x1=max_val, y1=max_val,
                      line=dict(dash='dash', color='gray', width=2), row=2, col=2)
        fig.update_geos(projection_type="robinson", showcoastlines=True, coastlinecolor="black")
        fig.update_layout(title_text="Dashboard Comparativo: Países Estudados × Países de Origem",
                          height=900, showlegend=False, font=dict(size=12))
        fig.update_xaxes(title_text="Contagem Estudo", row=2, col=2)
        fig.update_yaxes(title_text="Contagem Origem", row=2, col=2)
        return fig

    # ==========================================================
    # 7️⃣ Gráfico de barras — Top países
    # ==========================================================
    def plot_top_countries(self, top_n: int = 15) -> go.Figure:
        if self.comparativo_df is None:
            raise ValueError("O dataframe comparativo ainda não foi criado. Execute create_comparison() primeiro.")

        def get_country_name(iso3):
            try:
                return pycountry.countries.get(alpha_3=iso3).name
            except:
                return iso3

        df = self.comparativo_df.copy()
        df['País_Nome'] = df['ISO3'].apply(get_country_name)
        top_paises = df.nlargest(top_n, 'Total').iloc[::-1]

        fig = go.Figure()
        fig.add_trace(go.Bar(
            y=top_paises['País_Nome'],
            x=top_paises['Contagem_Estudo'],
            name='Estudados',
            orientation='h',
            marker=dict(color='#E63946', line=dict(color='#8B0000', width=1))
        ))
        fig.add_trace(go.Bar(
            y=top_paises['País_Nome'],
            x=top_paises['Contagem_Origem'],
            name='Origem',
            orientation='h',
            marker=dict(color='#457B9D', line=dict(color='#1D3557', width=1))
        ))
        fig.update_layout(
            title=f'<b>Top {top_n} Países — Estudos vs Origem de Publicações</b>',
            xaxis_title='Número de Artigos',
            yaxis_title='',
            barmode='group',
            height=600,
            template='plotly_white',
            font=dict(size=12)
        )
        return fig

# ==========================================================
# 8️⃣ Execução principal
# ==========================================================
def main():
    # 📂 Caminho do arquivo CSV de entrada (ajuste conforme necessário)
    CSV_PATH = "/content/drive/MyDrive/geo_2/final_geo2.csv"

    analyzer = CountryAnalysisDashboard(CSV_PATH)
    analyzer.load_data()
    analyzer.process_studied_countries()
    analyzer.process_origin_countries()
    analyzer.create_comparison()
    analyzer.print_summary()

    # Criar e mostrar gráficos
    dashboard = analyzer.create_dashboard()
    dashboard.show()
    fig_top = analyzer.plot_top_countries(top_n=15)
    fig_top.show()


if __name__ == "__main__":
    main()


In [ ]:
# ============================================================
# 📊 VISUALIZAÇÕES COM BASE NAS NOVAS COLUNAS
# ============================================================
import pandas as pd
import plotly.express as px
import matplotlib.pyplot as plt

# Caminho do CSV processado
csv_path = "/content/drive/MyDrive/geo_2/localizacoes_processadas_melhorado.csv"

# Carregar o dataset
df = pd.read_csv(csv_path)
print(f"📥 Dataset carregado com {len(df)} entradas.")

# Converter colunas numéricas (caso tenham vindo como string)
df['num_paises'] = pd.to_numeric(df['num_paises'], errors='coerce').fillna(0).astype(int)
df['num_instituicoes'] = pd.to_numeric(df['num_instituicoes'], errors='coerce').fillna(0).astype(int)

# ============================================================
# 1️⃣ Distribuição de número de países por artigo
# ============================================================
fig1 = px.histogram(
    df,
    x='num_paises',
    nbins=10,
    title="📈 Distribuição do número de países por artigo",
    labels={'num_paises': 'Número de países'},
    color_discrete_sequence=['#4C72B0']
)
fig1.update_layout(bargap=0.2)
fig1.show()

# ============================================================
# 2️⃣ Distribuição do número de instituições por artigo
# ============================================================
fig2 = px.histogram(
    df,
    x='num_instituicoes',
    nbins=20,
    title="🏫 Distribuição do número de instituições por artigo",
    labels={'num_instituicoes': 'Número de instituições'},
    color_discrete_sequence=['#55A868']
)
fig2.update_layout(bargap=0.2)
fig2.show()

# ============================================================
# 3️⃣ Proporção de artigos com múltiplos países
# ============================================================
prop_multiplos = df['tem_multiplos_paises'].value_counts(normalize=True) * 100
fig3 = px.pie(
    names=prop_multiplos.index.map({True: 'Múltiplos países', False: 'Apenas um país'}),
    values=prop_multiplos.values,
    title="🌍 Proporção de artigos com múltiplos países"
)
fig3.show()

# ============================================================
# 4️⃣ Top países mais frequentes
# ============================================================
# Separar e contar países ISO3
todos_paises = df['iso3_paises'].dropna().str.split(';')
todos_paises = [p.strip() for sublist in todos_paises for p in sublist if p.strip() != ""]
freq_paises = pd.Series(todos_paises).value_counts().reset_index()
freq_paises.columns = ['ISO3', 'Frequência']

# Gráfico dos 15 principais
top15 = freq_paises.head(15)
fig4 = px.bar(
    top15,
    x='ISO3',
    y='Frequência',
    title='🌎 Top 15 países mais frequentes nos artigos',
    text='Frequência',
    color='Frequência',
    color_continuous_scale='Blues'
)
fig4.update_traces(textposition='outside')
fig4.update_layout(xaxis_title='País (ISO3)', yaxis_title='Frequência')
fig4.show()

# ============================================================
# 5️⃣ Relação entre países e número de instituições
# ============================================================
fig5 = px.scatter(
    df,
    x='num_paises',
    y='num_instituicoes',
    title='🔍 Relação entre número de países e número de instituições por artigo',
    labels={'num_paises': 'Nº de Países', 'num_instituicoes': 'Nº de Instituições'},
    color='tem_multiplos_paises',
    color_discrete_map={True: 'red', False: 'blue'},
    opacity=0.7
)
fig5.update_traces(marker=dict(size=10))
fig5.show()

# ============================================================
# 6️⃣ Diagnóstico rápido no console
# ============================================================
print("\n📊 Diagnóstico rápido:")
print(f"➡️ Total de artigos: {len(df)}")
print(f"➡️ Artigos com múltiplos países: {df['tem_multiplos_paises'].sum()} ({df['tem_multiplos_paises'].mean()*100:.1f}%)")
print(f"➡️ Média de países por artigo: {df['num_paises'].mean():.2f}")
print(f"➡️ Média de instituições por artigo: {df['num_instituicoes'].mean():.2f}")
print(f"➡️ Total de países únicos identificados: {len(freq_paises)}")


In [ ]:
# === HEATMAP VISUALIZATION ===

import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

df = pd.read_csv("/content/drive/MyDrive/geo_2/final_geo2.csv")

# -------------------------------------------------
# 1. NORMALIZAR CONTINENTES (SEM NAN!)
# -------------------------------------------------

# Tratar NA corretamente ANTES de transformar em string
df["País_continente"] = df["País_continente"].fillna("")

# Converter para lista de continentes
df["País_continente"] = (
    df["País_continente"]
    .apply(lambda x: str(x))                       # garante string
    .apply(lambda x: x.split("/"))                  # divide
    .apply(lambda lst: [c.strip().title() for c in lst if c.strip() not in ["", "Nan", "Na", "None"]])
)

# Explode
df = df.explode("País_continente")

# Remover linhas vazias
df = df[df["País_continente"] != ""]

# -------------------------------------------------
# 2. MAPEAR TÓPICOS PARA OS NOMES DO dicionário
# -------------------------------------------------

megatopicos_dict = {
    1: ["T1 – IPV in Multiple Contexts", "T4 – Domestic Violence & Culture",
        "T11 – Pregnancy & Childbirth", "T26 – Norms & Attitudes Toward IPV",
        "T30 – IPV Prevalence & Determinants"],
    2: ["T2 – Adolescent & Youth Violence", "T15 – Educational Settings",
        "T23 – Dating Violence & Youth Victimization", "T29 – Childhood Maltreatment"],
    3: ["T3 – Sex Workers & Trafficking", "T10 – People with Disabilities",
        "T12 – Elder Abuse & Neglect", "T13 – Drug Use & Urban Marginalization",
        "T16 – Sexual, Gender & Ethnic Minorities", "T18 – Economic Interventions & Empowerment"],
    4: ["T17 – Migrants & Asylum Seekers", "T24 – War/Armed Conflict",
        "T25 – Humanitarian Services"],
    5: ["T5 – COVID-19", "T7 – Survivor Experiences & Community Response",
        "T8 – HIV Contexts", "T31 – Trauma & Mental Health",
        "T32 – Reproductive Health, Abortion & Contraception"],
    6: ["T6 – Workplace Violence", "T9 – Emergency Services & Police Contexts",
        "T19 – Sexual Harassment & Institutional", "T21 – Global & Structural Patterns",
        "T27 – Community & Neighborhood Context", "T28 – Spatial Distribution of Lethal Violence"],
    7: ["T14 – GBV & Female Mortality in Brazil",
        "T20 – IPV Homicides", "T22 – IPV Sexual Violence"]
}


# Criar dicionário invertido: { número : nome }
topic_name_map = {}

for mega, lista in megatopicos_dict.items():
    for item in lista:
        # Extrair número T6, T7 etc
        number = int(item.split("–")[0].replace("T", "").strip())
        topic_name_map[number] = item.strip()

# Mapear coluna Topico (números) para nomes completos
df["Topico_nome"] = df["Topico"].map(topic_name_map)

# Ordenar os tópicos exatamente na ordem dos megatópicos
topic_order = [name for lst in megatopicos_dict.values() for name in lst]

# -------------------------------------------------
# 3. Criar tabela cruzada continente × tópico (ordenada)
# -------------------------------------------------

table_topics = pd.crosstab(df["País_continente"], df["Topico_nome"])
table_topics = table_topics[topic_order]  # reordena colunas

# -------------------------------------------------
# 4. Criar Heatmap
# -------------------------------------------------

fig, ax = plt.subplots(figsize=(16, 7.5))
mask_zero = table_topics == 0

sns.heatmap(
    table_topics,
    cmap="YlGnBu",
    linewidths=1,
    linecolor="#EAEAEA",
    cbar_kws={"label": "Frequency", "shrink": 0.8},
    annot=True,
    fmt=".0f",
    mask=mask_zero,
    vmin=0,
    ax=ax
)

# Slashes nas células zero
for i in range(table_topics.shape[0]):
    for j in range(table_topics.shape[1]):
        if table_topics.iloc[i, j] == 0:
            ax.text(j + 0.5, i + 0.5, "/", ha="center", va="center",
                    color="gray", fontsize=11, fontweight="bold", alpha=0.6)

# Labels
ax.set_xticklabels(ax.get_xticklabels(), rotation=-35, ha="left", fontsize=11.5)
ax.set_yticklabels(ax.get_yticklabels(), fontsize=11.5, fontweight="semibold")
ax.set_xlabel("Topic", fontweight="bold", fontsize=11)
ax.set_ylabel("Continent", fontweight="bold", fontsize=11)
plt.tight_layout()
plt.savefig("fig1_heatmap_topics.png", dpi=300, bbox_inches="tight", facecolor="white")
plt.show()
print("\n==============================================")
print("📊 TABELA USADA NO HEATMAP (Continente × Tópico)")
print("==============================================")
print(table_topics.to_string())


In [ ]:
# 📊 Professional Table of Topic Prevalence by Continent

import pandas as pd
import matplotlib.pyplot as plt

# ============================
# 🔢 TOP 3 TOPICOS POR CONTINENTE, CALCULADO DO CONJUNTO
# ============================
# ANTES esta celula trazia a lista escrita a mao, com os valores de quando a
# figura foi feita. Tres das cinco linhas divergiam do conjunto auditado, e a
# linha da Africa contradizia a figura de topicos por continente, que fica ao
# lado no mesmo documento. Os nomes tambem eram parafrases, e um deles, "Youth
# violence" para o T23, e praticamente o nome do T2.
#
# CRITERIO DE DESEMPATE, declarado porque ha empate em quatro dos cinco
# continentes: vence o topico maior no corpus inteiro e, persistindo, o de
# menor numero.

base = pd.read_csv("/content/drive/MyDrive/geo_2/final_geo2.csv")

# Nomes canonicos, com o codigo, a partir do megatopicos_dict
topic_map = {}
for _itens in megatopicos_dict.values():
    for _item in _itens:
        _n = int(_item.split("–")[0].replace("T", "").strip())
        topic_map[_n] = _item.strip()

tamanho_no_corpus = base["Topico"].value_counts()

_d = base.copy()
_d["País_continente"] = _d["País_continente"].fillna("").astype(str)
_d["País_continente"] = _d["País_continente"].apply(lambda x: x.split("/"))
_d = _d.explode("País_continente")
_d["País_continente"] = _d["País_continente"].str.strip()
_d = _d[_d["País_continente"] != ""]

nome_en = {"América": "America", "Europa": "Europe", "Ásia": "Asia",
           "África": "Africa", "Oceania": "Oceania"}

rows = []
for _cont in ["América", "Europa", "Ásia", "África", "Oceania"]:
    _vc = _d[_d["País_continente"] == _cont]["Topico"].value_counts()
    if _vc.empty:
        continue
    _rank = sorted(
        _vc.items(),
        key=lambda kv: (-kv[1], -int(tamanho_no_corpus.get(kv[0], 0)), kv[0]))
    _top = [topic_map.get(t, "T%d" % t) for t, _ in _rank[:3]]
    rows.append({
        "Continent": nome_en.get(_cont, _cont),
        "1st Predominant Topic": _top[0],
        "2nd Topic": _top[1],
        "3rd Topic": _top[2],
    })

df = pd.DataFrame(rows)

# ============================
# 🎨 PROFESSIONAL DESIGN
# ============================
# Mais largo: com o codigo do topico junto do nome, o rotulo mais longo
# passava da borda da ultima coluna.
fig, ax = plt.subplots(figsize=(19, 6))
ax.axis("tight")
ax.axis("off")

# Professional palette: gray tones + corporate blue
header_color = "#1F3A70"  # Dark corporate blue
alt_colors = ["#F8F9FA", "#FFFFFF"]  # Alternating row colors
ranking_colors = ["#2E5090", "#4A7BA7", "#6B9BC3"]  # Blue gradient for ranking

# Create table
table = ax.table(
    cellText=df.values,
    colLabels=df.columns,
    cellLoc="left",
    loc="center",
    colWidths=[0.12, 0.29, 0.29, 0.29],
    bbox=[0, 0, 1, 1]
)

table.auto_set_font_size(False)
table.set_fontsize(20)
table.scale(1, 2.5)

# Header styling
for i in range(len(df.columns)):
    cell = table[(0, i)]
    cell.set_facecolor(header_color)
    cell.set_text_props(weight="bold", color="white", size=14)
    cell.set_height(0.08)
    cell.set_edgecolor("white")
    cell.set_linewidth(2)

# Data row styling
for i in range(len(df)):
    # Continent column
    cell = table[(i + 1, 0)]
    cell.set_facecolor("#E8EEF5")
    cell.set_text_props(weight="bold", size=14, color="#1F3A70")
    cell.set_edgecolor("#D0D8E8")
    cell.set_linewidth(1.5)

    # Topic columns with colored borders
    for j in range(1, len(df.columns)):
        cell = table[(i + 1, j)]

        # Subtle alternating background
        cell.set_facecolor(alt_colors[i % 2])

        # Left border color by ranking
        cell.set_edgecolor(ranking_colors[j - 1])
        cell.set_linewidth(3)

        # Text properties
        cell.set_text_props(size=10, color="#2C3E50")
        cell.set_height(0.09)

# Add title and note


plt.subplots_adjust(top=0.93, bottom=0.08)
plt.savefig("table_topics_by_continent.png", dpi=300, bbox_inches="tight", facecolor="white")
plt.show()

print("\n✅ Table saved as 'table_topics_by_continent.png'")
print("\n" + "="*80)
print("DATA SUMMARY")
print("="*80)
print(df.to_string(index=False))


In [ ]:
# 📊 Heatmap: Relationship Between Production and Studied Continents

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# ============================
# 🔢 DATA
# ============================
data = {
    "Production Continent (Origin)": ["AMERICA", "EUROPE", "OCEANIA", "AFRICA", "ASIA"],
    "AMERICA": [141, 19, 2, 4, 5],
    "EUROPE": [19, 51, 1, 1, 5],
    "OCEANIA": [7, 4, 7, 1, 1],
    "AFRICA": [60, 25, 1, 95, 20],
    "ASIA": [35, 24, 4, 12, 68],
}
df = pd.DataFrame(data)
df.set_index("Production Continent (Origin)", inplace=True)

# ============================
# 🎨 PROFESSIONAL DESIGN
# ============================
plt.figure(figsize=(8, 6))
ax = sns.heatmap(
    df,
    annot=True,
    fmt="d",
    cmap="Blues",
    linewidths=0.8,
    linecolor="#D0D8E8",
    cbar_kws={"label": "Number of Studies"}
)

# === STYLING ===
ax.set_xlabel("DESTINATION (STUDY FOCUS)", fontsize=12, fontweight="bold", color="#1F3A70", labelpad=12)
ax.set_ylabel("ORIGIN (PRODUCTION)", fontsize=12, fontweight="bold", color="#1F3A70", labelpad=12)

ax.set_xticklabels(ax.get_xticklabels(), rotation=0, fontsize=10, color="#2C3E50")
ax.set_yticklabels(ax.get_yticklabels(), rotation=0, fontsize=10, color="#2C3E50")


# Nota de rodapé
plt.figtext(
    0.5, -0.03,
    "Note: Values represent the number of studies produced in one continent focusing on another.",
    ha="center",
    fontsize=9.5,
    style="italic",
    color="#666666"
)

plt.tight_layout()
plt.savefig("heatmap_production_vs_study.png", dpi=300, bbox_inches="tight", facecolor="white")
plt.show()




In [ ]:
# 📊 Z-Scores (Standardized Residuals) for Origin × Study Focus Matrix

import pandas as pd
import numpy as np
import scipy.stats as stats
import seaborn as sns
import matplotlib.pyplot as plt

# ============================
# 🔢 ORIGINAL DATA
# ============================
data = {
    "Production Continent (Origin)": ["AMERICA", "EUROPE", "OCEANIA", "AFRICA", "ASIA"],
    "AMERICA": [141, 19, 2, 4, 5],
    "EUROPE": [19, 51, 1, 1, 5],
    "OCEANIA": [7, 4, 7, 1, 1],
    "AFRICA": [60, 25, 1, 95, 20],
    "ASIA": [35, 24, 4, 12, 68],
}

df = pd.DataFrame(data).set_index("Production Continent (Origin)")

# ============================
# 🧮 CHI-SQUARE + EXPECTED VALUES
# ============================
chi2, p, dof, expected = stats.chi2_contingency(df)

expected_df = pd.DataFrame(expected, index=df.index, columns=df.columns)

# ============================
# 🧮 STANDARDIZED RESIDUALS (Z-SCORES)
# ============================
z_scores = (df - expected_df) / np.sqrt(expected_df)

print("\n=== STANDARDIZED RESIDUALS (Z-Scores) ===\n")
print(z_scores.round(2))

# ============================
# 🎨 HEATMAP OF Z-SCORES
# ============================
plt.figure(figsize=(8, 6))

ax = sns.heatmap(
    z_scores,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",          # red = above expected, blue = below
    center=0,
    vmin=-5, vmax=5,          # keeps colors comparable across cells
    linewidths=0.8,
    linecolor="#D0D8E8",
    cbar_kws={"label": "Standardized Residual (Z-Score)"}
)

# === LABELS ===
ax.set_xlabel("DESTINATION (STUDY FOCUS)", fontsize=12, fontweight="bold", color="#1F3A70", labelpad=12)
ax.set_ylabel("ORIGIN (PRODUCTION)", fontsize=12, fontweight="bold", color="#1F3A70", labelpad=12)

ax.set_xticklabels(ax.get_xticklabels(), rotation=0, fontsize=10, color="#2C3E50")
ax.set_yticklabels(ax.get_yticklabels(), rotation=0, fontsize=10, color="#2C3E50")

# === FOOTER ===
plt.figtext(
    0.5, -0.03,
    "Note: Positive values indicate flows ABOVE expected; negative values indicate flows BELOW expected.",
    ha="center",
    fontsize=9.5,
    style="italic",
    color="#666666"
)

plt.tight_layout()
plt.savefig("heatmap_zscores_fluxos.png", dpi=300, bbox_inches="tight", facecolor="white")
plt.show()


In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import chi2

# ==========================================================
# 1. MATRIZ ORIGINAL
# ==========================================================
df = pd.DataFrame(
    {
        "AMERICA":  [141, 19, 2, 4, 5],
        "EUROPE":   [19, 51, 1, 1, 5],
        "OCEANIA":  [7, 4, 7, 1, 1],
        "AFRICA":   [60, 25, 1, 95, 20],
        "ASIA":     [35, 24, 4, 12, 68]
    },
    index=["AMERICA", "EUROPE", "OCEANIA", "AFRICA", "ASIA"]
)

# ==========================================================
# 2. REMOVER DIAGONAL — tanto observada quanto do modelo
# ==========================================================
df_no_diag = df.copy()
np.fill_diagonal(df_no_diag.values, 0)

# Totais sem contar diagonal
row_totals = df_no_diag.sum(axis=1)
col_totals = df_no_diag.sum(axis=0)
grand_total = df_no_diag.values.sum()

# ==========================================================
# 3. MATRIZ ESPERADA SEM DIAGONAL
# ==========================================================
expected = np.outer(row_totals, col_totals) / grand_total
expected_df = pd.DataFrame(expected, index=df.index, columns=df.columns)

# Zerar diagonal também na esperada
np.fill_diagonal(expected_df.values, 0)

# ==========================================================
# 4. RESÍDUOS PADRONIZADOS
# ==========================================================
z = (df_no_diag - expected_df) / np.sqrt(expected_df)

print("\n=== STANDARDIZED RESIDUALS (Z-Scores) ===\n")
print(z.round(2))

# ==========================================================
# 5. HEATMAP FINAL
# ==========================================================
plt.figure(figsize=(9,7))
sns.heatmap(z, annot=True, fmt=".2f", cmap="coolwarm", center=0,
            linewidths=0.6, linecolor="white")
plt.title("Standardized Residuals (Diagonal Removed)", fontsize=14)
plt.show()

# ==========================================================
# 6. QUI-QUADRADO MANUAL
# ==========================================================
chi2_value = ((df_no_diag - expected_df)**2 / expected_df).sum().sum()

# Graus de liberdade corretos:
# df = (r - 1) * (c - 1)  — mesmo removendo diagonal
# (a remoção da diagonal altera os valores, não a estrutura)
r, c = df_no_diag.shape
df_chi = (r - 1) * (c - 1)

print("\nChi-square (manual, diagonal removed):", chi2_value)
print("Degrees of freedom:", df_chi)
print("p-value:", 1 - chi2.cdf(chi2_value, df_chi))


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# ------------------------------------------------------------
# 1. Títulos dos megatópicos
# ------------------------------------------------------------
megatopicos_titles = {
    1: "M1 – Intimate partner and domestic violence",
    2: "M2 – Children, adolescents and youth",
    3: "M3 – Minority Groups and Vulnerable Populations",
    4: "M4 – Humanitarian, migration and armed conflicts",
    5: "M5 – Locations and contextual factors",
    6: "M6 – Health and social support",
    7: "M7 – Lethal and Sexual Violence"
}

# ------------------------------------------------------------
# 2. Lista de tópicos por megatópico
# ------------------------------------------------------------
megatopicos_dict = {
    1: ["T1 – IPV in Multiple Contexts", "T4 – Domestic Violence & Culture",
        "T11 – Pregnancy & Childbirth", "T26 – Norms & Attitudes Toward IPV",
        "T30 – IPV Prevalence & Determinants"],
    2: ["T2 – Adolescent & Youth Violence", "T15 – Educational Settings",
        "T23 – Dating Violence & Youth Victimization", "T29 – Childhood Maltreatment"],
    3: ["T3 – Sex Workers & Trafficking", "T10 – People with Disabilities",
        "T12 – Elder Abuse & Neglect", "T13 – Drug Use & Urban Marginalization",
        "T16 – Sexual, Gender & Ethnic Minorities", "T18 – Economic Interventions & Empowerment"],
    4: ["T17 – Migrants & Asylum Seekers", "T24 – War/Armed Conflict",
        "T25 – Humanitarian Services"],
    5: ["T5 – COVID-19", "T7 – Survivor Experiences & Community Response",
        "T8 – HIV Contexts", "T31 – Trauma & Mental Health",
        "T32 – Reproductive Health, Abortion & Contraception"],
    6: ["T6 – Workplace Violence", "T9 – Emergency Services & Police Contexts",
        "T19 – Sexual Harassment & Institutional", "T21 – Global & Structural Patterns",
        "T27 – Community & Neighborhood Context", "T28 – Spatial Distribution of Lethal Violence"],
    7: ["T14 – GBV & Female Mortality in Brazil",
        "T20 – IPV Homicides", "T22 – IPV Sexual Violence"]
}

# ------------------------------------------------------------
# 3. Criar tabela com 1 linha por tópico
# ------------------------------------------------------------
rows = []
megatopic_spans = []   # para saber onde começam e terminam os blocos
start_row = 1          # linha 0 será o cabeçalho

for mt, topics in megatopicos_dict.items():
    megatopic_label = f"M{mt} – {megatopicos_titles[mt]}"
    end_row = start_row + len(topics) - 1

    # salvar intervalo para o "fake rowspan"
    megatopic_spans.append((start_row, end_row, megatopic_label))

    # tópicos
    for t in topics:
        rows.append(["", t])  # megatopic preenchido depois
    start_row = end_row + 1

df_table = pd.DataFrame(rows, columns=["Megatopic", "Topic"])

# ------------------------------------------------------------
# 4. Plotar tabela com fake rowspan
# ------------------------------------------------------------
fig, ax = plt.subplots(figsize=(20, 14))
ax.axis("off")

table = ax.table(
    cellText=df_table.values,
    colLabels=df_table.columns,
    cellLoc='left',
    loc='center'
)

table.auto_set_font_size(False)
table.set_fontsize(11)
table.scale(1, 1.5)

# ------------------------------------------------------------
# 5. Estilo geral
# ------------------------------------------------------------
for (row, col), cell in table.get_celld().items():
    if row == 0:  # cabeçalho
        cell.set_facecolor("#1F3A70")
        cell.set_text_props(color="white", fontweight="bold")
    else:
        cell.set_facecolor("#F7F9FC")

# ------------------------------------------------------------
# 6. Aplicar o FAKE ROWSPAN
# ------------------------------------------------------------
for start, end, label in megatopic_spans:
    # primeira linha do bloco: texto + fundo forte
    first_cell = table[start, 0]
    first_cell.set_text_props(text=label, fontweight='bold')
    first_cell.set_facecolor("#DDE7F2")

    # linhas intermediárias do bloco
    for row in range(start + 1, end + 1):
        cell = table[row, 0]
        cell.get_text().set_text("")      # remove texto
        cell.set_facecolor("#DDE7F2")     # mesmo fundo do bloco
        cell.visible_edges = "LR"         # remove bordas superior e inferior

plt.title(
    "Megatopics and Topics (Grouped with Fake Rowspan)",
    fontsize=16,
    fontweight="bold",
    color="#1F3A70",
    pad=20
)

plt.tight_layout()
plt.savefig("/content/megatopics_grouped_fake_rowspan.png", dpi=300, bbox_inches="tight")
plt.show()

print("Tabela salva como: /content/megatopics_grouped_fake_rowspan.png")


In [ ]:
# ============================================================
# 🎨 CONFIGURAÇÃO DE DESIGN PADRÃO
# ============================================================
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams["font.family"] = "serif"

TITLE_COLOR = "#1F3A70"
BAR_COLOR = "#1F3A70"
GRID_COLOR = "#7F7F7F"


# ============================================================
# 📌 FUNÇÃO PARA NORMALIZAR COLUNAS DE CONTINENTES
# ============================================================
def normalizar_continentes(df, orig_col, norm_col):
    """
    Divide entradas com "/" e limpa valores inconsistentes.
    Cria uma coluna explosiva com 1 continente por linha.
    """
    df[orig_col] = df[orig_col].fillna("")

    df[norm_col] = (
        df[orig_col]
        .astype(str)
        .str.split("/")
        .apply(lambda lst: [
            c.strip().title()
            for c in lst
            if c.strip() not in ["", "Nan", "Na", "None"]
        ])
    )

    df = df.explode(norm_col)
    df = df[df[norm_col].notna() & (df[norm_col] != "")]
    return df


# ============================================================
# 📂 1. CARREGAR CSV
# ============================================================
df = pd.read_csv("/content/drive/MyDrive/geo_2/final_geo2.csv")


# ============================================================
# 🌍 2. NORMALIZAR CONTINENTES — ORIGEM E LOCALIZAÇÃO
# ============================================================

# Origem do autor
df1 = normalizar_continentes(df.copy(), "País_continente", "continente_origem")

# Localização do estudo
df2 = normalizar_continentes(df.copy(), "pais_location_continente", "continente_local")


# ============================================================
# 📊 3. CONTAGEM POR CONTINENTE
# ============================================================
cont_origem = df1["continente_origem"].value_counts()
cont_local  = df2["continente_local"].value_counts()


# ============================================================
# 📈 4. GRÁFICOS — ORIGEM E LOCALIZAÇÃO
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(18, 6))
fig.patch.set_facecolor("white")

# ----------------- GRÁFICO 1 – Origem -----------------------
axes[0].bar(cont_origem.index, cont_origem.values, color=BAR_COLOR, alpha=0.85)

axes[0].set_title(
    "Distribution of Study Production by Continent (Origin)",
    fontsize=18, fontweight="bold", color=TITLE_COLOR, pad=12
)

axes[0].set_xlabel("Continent", fontsize=13, fontweight="bold")
axes[0].set_ylabel("Number of Studies", fontsize=13, fontweight="bold")

axes[0].grid(axis="y", linestyle="--", color=GRID_COLOR, alpha=0.4)


# ----------------- GRÁFICO 2 – Localização -----------------------
axes[1].bar(cont_local.index, cont_local.values, color=BAR_COLOR, alpha=0.85)

axes[1].set_title(
    "Distribution of Study Focus by Continent (Location)",
    fontsize=18, fontweight="bold", color=TITLE_COLOR, pad=12
)

axes[1].set_xlabel("Continent", fontsize=13, fontweight="bold")
axes[1].set_ylabel("Number of Studies", fontsize=13, fontweight="bold")

axes[1].grid(axis="y", linestyle="--", color=GRID_COLOR, alpha=0.4)


plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# --------------------------------------------------
# 🎨 1. DESIGN TOKENS
# --------------------------------------------------
plt.rcParams["font.family"] = "serif"

MEGA_COLORS = [
    "#1F3A70", "#E85C91", "#3C9D9B",
    "#F1B24A", "#8C6AA0", "#FF7F50", "#7F7F7F"
]

TITLE_COLOR = "#1F3A70"
EDGE_COLOR = "white"
EDGE_WIDTH = 1.1
BAR_ALPHA = 0.9

# --------------------------------------------------
# 2. LOAD CSV
# --------------------------------------------------
df = pd.read_csv("/content/drive/MyDrive/geo_2/final_geo2.csv")

# --------------------------------------------------
# 3. NORMALIZATION FUNCTION
# --------------------------------------------------
def normalize_continents(df, orig_col, norm_col):
    df[orig_col] = df[orig_col].fillna("")
    df[norm_col] = (
        df[orig_col]
        .astype(str)
        .str.split("/")
        .apply(lambda lst: [
            c.strip().title()
            for c in lst
            if c.strip() not in ["", "Nan", "Na", "None"]
        ])
    )
    df = df.explode(norm_col)
    df = df[df[norm_col].notna() & (df[norm_col] != "")]
    return df

# --------------------------------------------------
# 4. NORMALIZE BOTH COLUMNS
# --------------------------------------------------
df_focus = normalize_continents(df.copy(), "País_continente", "focus_continent")
df_origin = normalize_continents(df.copy(), "pais_location_continente", "origin_continent")

# --------------------------------------------------
# 5. RAW COUNTS AND PERCENTAGES
# --------------------------------------------------
raw_focus = df_focus["focus_continent"].value_counts().sort_index()
raw_origin = df_origin["origin_continent"].value_counts().sort_index()

pct_focus = (raw_focus / raw_focus.sum() * 100).round(2)
pct_origin = (raw_origin / raw_origin.sum() * 100).round(2)

# Print values used in the chart
print("\n=== RAW COUNTS — Study Focus Continent ===")
print(raw_focus)

print("\n=== RAW COUNTS — First Author Origin Continent ===")
print(raw_origin)

print("\n=== PERCENTAGES — Study Focus Continent (%) ===")
print(pct_focus)

print("\n=== PERCENTAGES — First Author Origin Continent (%) ===")
print(pct_origin)

# --------------------------------------------------
# 6. ALIGN CONTINENTS ACROSS BOTH VARIABLES
# --------------------------------------------------
continents = sorted(set(raw_focus.index).union(set(raw_origin.index)))

values_focus = [pct_focus.get(c, 0) for c in continents]
values_origin = [pct_origin.get(c, 0) for c in continents]

# --------------------------------------------------
# 7. BAR CHART (SIDE-BY-SIDE)
# --------------------------------------------------
x = np.arange(len(continents))
w = 0.35

plt.figure(figsize=(12, 7))

plt.bar(
    x - w/2, values_focus,
    width=w,
    label="Study Focus Continent",
    color=MEGA_COLORS[0],
    alpha=BAR_ALPHA,
    edgecolor=EDGE_COLOR,
    linewidth=EDGE_WIDTH
)

plt.bar(
    x + w/2, values_origin,
    width=w,
    label="First Author Origin Continent",
    color=MEGA_COLORS[1],
    alpha=BAR_ALPHA,
    edgecolor=EDGE_COLOR,
    linewidth=EDGE_WIDTH
)

# --------------------------------------------------
# 8. STYLING
# --------------------------------------------------
plt.xticks(x, continents, rotation=30, fontsize=11)
plt.ylabel("Percentage (%)", fontsize=12)

plt.legend(frameon=False)
plt.grid(axis="y", linestyle="--", alpha=0.4)

plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# --------------------------------------------------
# 🎨 1. DESIGN TOKENS
# --------------------------------------------------
plt.rcParams["font.family"] = "serif"

MEGA_COLORS = [
    "#1F3A70", "#E85C91", "#3C9D9B",
    "#F1B24A", "#8C6AA0", "#FF7F50", "#7F7F7F"
]

TITLE_COLOR = "#1F3A70"

BAR_ALPHA = 0.90
EDGE_COLOR = "white"
EDGE_WIDTH = 1.1

# --------------------------------------------------
# 📂 2. CARREGAR DADOS
# --------------------------------------------------
df = pd.read_csv("/content/drive/MyDrive/geo_2/final_geo2.csv")

# --------------------------------------------------
# 🔧 3. FUNÇÃO PARA NORMALIZAR CONTINENTES
# --------------------------------------------------
def normalizar_continentes(df, orig_col, norm_col):
    df[orig_col] = df[orig_col].fillna("")

    df[norm_col] = (
        df[orig_col]
        .astype(str)
        .str.split("/")
        .apply(lambda lst: [
            c.strip().title()
            for c in lst
            if c.strip() not in ["", "Nan", "Na", "None"]
        ])
    )

    df = df.explode(norm_col)
    df = df[df[norm_col].notna() & (df[norm_col] != "")]
    return df

# --------------------------------------------------
# 🧹 4. NORMALIZAR AS DUAS COLUNAS
# --------------------------------------------------
df1 = normalizar_continentes(df.copy(), "País_continente", "continente_norm1")
df2 = normalizar_continentes(df.copy(), "pais_location_continente", "continente_norm2")

# --------------------------------------------------
# 📊 5. CONTAGENS
# --------------------------------------------------
contagem1 = df1["continente_norm1"].value_counts()
contagem2 = df2["continente_norm2"].value_counts()

# --------------------------------------------------
# 🥧 6. GRÁFICO 1 — País_continente
# --------------------------------------------------
plt.figure(figsize=(8, 8))
plt.pie(
    contagem1.values,
    labels=contagem1.index,
    autopct="%1.1f%%",
    colors=MEGA_COLORS[:len(contagem1)],
    startangle=90,
    wedgeprops={"edgecolor": EDGE_COLOR, "linewidth": EDGE_WIDTH}
)
plt.title("Distribution of Study Focus by Continent (Location)", fontsize=16, color=TITLE_COLOR)
plt.tight_layout()
plt.show()

# --------------------------------------------------
# 🥧 7. GRÁFICO 2 — pais_location_continente
# --------------------------------------------------
plt.figure(figsize=(8, 8))
plt.pie(
    contagem2.values,
    labels=contagem2.index,
    autopct="%1.1f%%",
    colors=MEGA_COLORS[:len(contagem2)],
    startangle=90,
    wedgeprops={"edgecolor": EDGE_COLOR, "linewidth": EDGE_WIDTH}
)
plt.title("Distribution of Study Production by Continent (Origin)", fontsize=16, color=TITLE_COLOR)
plt.tight_layout()
plt.show()


In [ ]:
# ==============================================================
# 🎨 DESIGN APLICADO — 100% STACKED BAR (Megatópicos × Continentes)
# ==============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

# --------------------------------------------------
# 1. Títulos dos Megatópicos
# --------------------------------------------------
megatopicos_titles = {
    1: "M1 – Intimate partner and domestic violence",
    2: "M2 – Children, adolescents and youth",
    3: "M3 – Minority Groups and Vulnerable Populations",
    4: "M4 – Humanitarian, migration and armed conflicts",
    5: "M5 – Health and social support",
    6: "M6 – Locations and contextual factors",
    7: "M7 – Lethal and Sexual Violence"
}

# --------------------------------------------------
# 🎨 2. DESIGN TOKENS
# --------------------------------------------------
plt.rcParams["font.family"] = "serif"

MEGA_COLORS = [
    "#1F3A70", "#E85C91", "#3C9D9B",
    "#F1B24A", "#8C6AA0", "#FF7F50", "#7F7F7F"
]

cor_por_megatopico = {
    mt: MEGA_COLORS[(mt-1) % len(MEGA_COLORS)]
    for mt in range(1, 8)
}

BAR_ALPHA = 0.90
EDGE_COLOR = "white"
EDGE_WIDTH = 1.1
TITLE_COLOR = "#1F3A70"

# --------------------------------------------------
# 3. Carregar dados
# --------------------------------------------------
df = pd.read_csv("/content/drive/MyDrive/geo_2/final_geo2.csv")

orig_col = "País_continente"
norm_col = "Continente_norm"

# --------------------------------------------------
# 4. Normalizar continentes (SEM NAN)
# --------------------------------------------------
df[orig_col] = df[orig_col].fillna("")

df[norm_col] = (
    df[orig_col]
    .astype(str)
    .str.split("/")
    .apply(lambda lst: [
        c.strip().title()
        for c in lst
        if c.strip() not in ["", "Nan", "Na", "None"]
    ])
)

df = df.explode(norm_col)
df = df[df[norm_col].notna() & (df[norm_col] != "")]

# --------------------------------------------------
# 5. Corrigir Megatópico (11 → 1)
# --------------------------------------------------
df["Megatopico"] = df["Megatopico"].replace({11: 1}).astype(int)

# --------------------------------------------------
# 6. Contagem (continente × megatópico)
# --------------------------------------------------
grouped = (
    df.groupby([norm_col, "Megatopico"])
      .size()
      .reset_index(name="n")
)

pivot = grouped.pivot(index=norm_col, columns="Megatopico", values="n").fillna(0)

# Percentuais finais usados no gráfico
percent = pivot.div(pivot.sum(axis=1).replace(0, np.nan), axis=0).fillna(0) * 100

cols_sorted = sorted(percent.columns)

# ==============================================================
# 📌 **PRINT DE TODOS OS VALORES USADOS NO GRÁFICO**
# ==============================================================

print("\n" + "="*80)
print("📌 VALORES BRUTOS (Continente × Megatópico)")
print("="*80)
print(pivot.to_string())
print("\n")

print("="*80)
print("📌 PERCENTUAIS (100% STACKED) — Estes são os valores usados no gráfico")
print("="*80)
print(percent.round(2).to_string())
print("\n")

print("="*80)
print("📌 CONTAGEM TOTAL POR CONTINENTE")
print("="*80)
print(pivot.sum(axis=1).to_string())
print("\n")

print("="*80)
print("📌 TITLES ORDENADOS DOS MEGATÓPICOS")
print("="*80)
for mt in cols_sorted:
    print(f"{mt}: {megatopicos_titles[mt]}")
print("\n")

# --------------------------------------------------
# 7. Plot
# --------------------------------------------------
fig, ax = plt.subplots(figsize=(12, 8))
fig.patch.set_facecolor("white")

bottom = np.zeros(len(percent))

for mt in cols_sorted:
    ax.bar(
        percent.index,
        percent[mt],
        bottom=bottom,
        label=megatopicos_titles[mt],
        color=cor_por_megatopico[mt],
        alpha=BAR_ALPHA,
        edgecolor=EDGE_COLOR,
        linewidth=EDGE_WIDTH
    )
    bottom += percent[mt].values

# --------------------------------------------------
# 🎨 8. Estilo Profissional
# --------------------------------------------------
ax.set_ylabel("Percentage (%)", fontsize=14, fontweight="bold", color=TITLE_COLOR)
ax.set_xlabel("Continent", fontsize=14, fontweight="bold", color=TITLE_COLOR)

ax.set_xticklabels(ax.get_xticklabels(), rotation=0, fontsize=11, color="#2C3E50")
ax.set_yticklabels(ax.get_yticklabels(), fontsize=11, color="#2C3E50")

ax.grid(axis="y", linestyle="--", linewidth=0.7, alpha=0.45)

legend = ax.legend(
    title="Megatopics",
    title_fontsize=13,
    fontsize=11,
    bbox_to_anchor=(1.05, 1),
    loc="upper left",
    frameon=True
)
legend.get_frame().set_facecolor("white")
legend.get_frame().set_edgecolor("#CCCCCC")
legend.get_frame().set_linewidth(0.8)

plt.tight_layout()
plt.savefig(
    "stacked_megatopics_by_continent.png",
    dpi=300,
    bbox_inches="tight",
    facecolor="white"
)

plt.show()


In [ ]:
# =======================================================
# 🌍 Visualização de Continentes e Sub-Regiões (12 Gráficos de Pizza)
# =======================================================
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

# =======================================================
# 1️⃣ Carregar CSV
# =======================================================
caminho_csv = "/content/drive/MyDrive/geo_2/final_geo2.csv"
df = pd.read_csv(caminho_csv)
print("📂 CSV carregado e colunas geográficas padronizadas.")

# =======================================================
# 2️⃣ Padronizar colunas geográficas
# =======================================================
geo_cols = ['País_continente', 'País_subregiao', 'pais_location_continente', 'pais_location_subregiao']
for col in geo_cols:
    df[col] = df[col].astype(str).str.strip().str.title()

# =======================================================
# 3️⃣ Função para separar múltiplos valores por '/'
# =======================================================
def separar_multiplos(coluna):
    registros = []
    for entrada in coluna.dropna():
        partes = [p.strip() for p in str(entrada).split('/') if p.strip()]
        registros.extend(partes)
    return pd.Series(registros)

# =======================================================
# 4️⃣ Gráfico de pizza comparando continentes (lado a lado)
# =======================================================
cont_estudo = separar_multiplos(df['País_continente']).value_counts().reset_index()
cont_estudo.columns = ['Continente', 'Estudo']

cont_origem = separar_multiplos(df['pais_location_continente']).value_counts().reset_index()
cont_origem.columns = ['Continente', 'Origem']

# Criar figura com subplots de pizza lado a lado
from plotly.subplots import make_subplots

fig = make_subplots(
    rows=1, cols=2,
    specs=[[{'type':'pie'}, {'type':'pie'}]],
    subplot_titles=('Continentes - Estudos', 'Continentes - Local de Origem')
)

fig.add_trace(go.Pie(labels=cont_estudo['Continente'], values=cont_estudo['Estudo'],
                     textinfo='label+percent', hole=0.3), row=1, col=1)

fig.add_trace(go.Pie(labels=cont_origem['Continente'], values=cont_origem['Origem'],
                     textinfo='label+percent', hole=0.3), row=1, col=2)

fig.update_layout(title_text='📊 Distribuição de Continentes (Estudo vs Local de Origem)')
fig.show()

# =======================================================
# 5️⃣ Normalizar sub-regiões compostas
# =======================================================
subregioes_cols = ['País_subregiao', 'pais_location_subregiao']

def normalizar_subregioes(df, coluna):
    df_exp = df[[coluna]].dropna().copy()
    df_exp[coluna] = df_exp[coluna].str.split('/').apply(lambda x: [p.strip() for p in x])
    df_exp = df_exp.explode(coluna).reset_index(drop=True)
    return df_exp

df_sub = {}
for col in subregioes_cols:
    df_sub[col] = normalizar_subregioes(df, col)

# =======================================================
# 6️⃣ Definir continentes
# =======================================================
continentes = {
    'América': ['América Do Norte', 'América Central', 'Caribe', 'América Do Sul'],
    'África': ['África Setentrional', 'África Ocidental', 'África Central', 'África Oriental', 'África Meridional'],
    'Europa': ['Europa Setentrional', 'Europa Ocidental', 'Europa Oriental', 'Europa Meridional'],
    'Oceania': ['Austrália E Nova Zelândia', 'Melanésia', 'Micronésia', 'Polinésia'],
    'Ásia': ['Ásia Central', 'Ásia Oriental', 'Ásia Meridional', 'Sudeste Asiático', 'Ásia Ocidental']
}

# =======================================================
# 7️⃣ Gráficos de pizza por sub-região (duas colunas cada continente)
# =======================================================
for col in subregioes_cols:
    print(f"\n🔹 Gerando gráficos de pizza para: {col}")
    for continente, regioes in continentes.items():
        # Filtrar sub-regiões do continente
        df_filtrado = df_sub[col][df_sub[col][col].isin(regioes)]
        if df_filtrado.empty:
            continue
        contagem = df_filtrado[col].value_counts().reset_index()
        contagem.columns = ['Sub-Região', 'Contagem']

        # Plotar gráfico de pizza
        fig = px.pie(
            contagem,
            names='Sub-Região',
            values='Contagem',
            title=f'📌 {continente} - Sub-Regiões ({col})',
            hole=0.3
        )
        fig.update_traces(textinfo='label+percent')
        fig.show()

# =======================================================
# 8️⃣ Resumo final
# =======================================================
print("="*70)
print("🌍 RESUMO FINAL DAS COLUNAS GEOGRÁFICAS")
print("="*70)
for col in geo_cols:
    print(f"\n📘 Coluna: '{col}'")
    contagem = df[col].value_counts(dropna=False)
    for valor, qtd in contagem.items():
        print(f"  - {valor}: {qtd}")
    print(f"Total de valores únicos: {df[col].nunique(dropna=False)}")
print("="*70)
print("✅ ANÁLISE FINAL COMPLETA")


In [ ]:
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.colors import LinearSegmentedColormap
import matplotlib.patches as mpatches
from collections import Counter

# --- Carregar dados do CSV ---
caminho_csv = "/content/drive/MyDrive/geo_2/final_geo.csv"

try:
    df = pd.read_csv(caminho_csv)
    print(f"✅ CSV carregado com sucesso: {len(df)} registros")

    if 'authors' not in df.columns:
        print("❌ Coluna 'authors' não encontrada no CSV")
        print(f"Colunas disponíveis: {list(df.columns)}")
        exit()

    artigos = df['authors'].dropna().tolist()
    print(f"📚 Artigos com autores válidos: {len(artigos)}")

    if len(artigos) == 0:
        print("❌ Nenhum autor encontrado na coluna 'authors'")
        exit()

except Exception as e:
    print(f"❌ Erro ao carregar CSV: {e}")
    exit()

# --- Função para limpar e processar autores ---
def processar_autores(string_autores):
    if pd.isna(string_autores):
        return []

    autores_str = str(string_autores).strip()
    separadores = [', and ', ' and ', ' , ', ';', '|', ',']

    for sep in separadores:
        if sep in autores_str:
            autores = [a.strip() for a in autores_str.split(sep)]
            autores = [a for a in autores if a and a != 'nan' and len(a) > 1]
            if len(autores) > 1:
                return autores

    return [autores_str] if autores_str and autores_str != 'nan' and len(autores_str) > 1 else []

# --- Criar grafo ---
G = nx.Graph()
artigos_processados = 0
autores_por_artigo = []

print("🔄 Processando autores...")

for artigo in artigos:
    autores = processar_autores(artigo)

    if len(autores) >= 2:
        artigos_processados += 1
        autores_por_artigo.append(autores)

        for i in range(len(autores)):
            for j in range(i+1, len(autores)):
                if autores[i] and autores[j]:
                    if G.has_edge(autores[i], autores[j]):
                        G[autores[i]][autores[j]]['weight'] += 1
                    else:
                        G.add_edge(autores[i], autores[j], weight=1)

print(f"✅ Processamento concluído:")
print(f"   • Artigos com colaboração: {artigos_processados}/{len(artigos)}")
print(f"   • Autores únicos: {G.number_of_nodes()}")
print(f"   • Conexões no grafo: {G.number_of_edges()}")

if G.number_of_nodes() == 0:
    print("❌ Nenhuma conexão criada. Verifique o formato dos dados.")
    exit()

# --- Análise da rede ---
degrees = dict(G.degree())
betweenness = nx.betweenness_centrality(G)
closeness = nx.closeness_centrality(G)
components = list(nx.connected_components(G))
largest_cc = max(components, key=len)
G_main = G.subgraph(largest_cc).copy()

print(f"\n🔍 Componente principal:")
print(f"   • Autores: {len(largest_cc)}")
print(f"   • Conexões: {G_main.number_of_edges()}")

# ============================================================
# VISUALIZAÇÃO 1: COMPONENTE PRINCIPAL (MELHORADA)
# ============================================================
fig = plt.figure(figsize=(18, 12))

degrees_main = dict(G_main.degree())
betweenness_main = nx.betweenness_centrality(G_main)

# Layout otimizado com força repulsiva melhor
print("🔄 Calculando layout...")
pos = nx.spring_layout(G_main, k=2.5, iterations=150, scale=2.5, seed=42)

# Tamanho dos nós: baseado no grau (número de colaborações)
min_size, max_size = 100, 1000
min_degree, max_degree = min(degrees_main.values()), max(degrees_main.values())
degrees_norm = [(degrees_main[n] - min_degree) / (max_degree - min_degree)
               for n in G_main.nodes()]
node_sizes = [min_size + (max_size - min_size) * d for d in degrees_norm]

# Cor dos nós: baseada em centralidade (importância na rede)
cmap = LinearSegmentedColormap.from_list('gradient', ['#E0E7FF', '#7C3AED'])
betweenness_vals = list(betweenness_main.values())
betweenness_min, betweenness_max = min(betweenness_vals), max(betweenness_vals)
betweenness_norm = [(b - betweenness_min) / (betweenness_max - betweenness_min)
                   for b in betweenness_vals]
node_colors = [cmap(b) for b in betweenness_norm]

# Espessura das arestas: baseada no peso (frequência de colaboração)
edge_weights = [G_main[u][v].get('weight', 1) for u, v in G_main.edges()]
max_weight = max(edge_weights)
edge_widths = [0.3 + 3.5 * (w / max_weight) for w in edge_weights]

# Desenho otimizado
ax = plt.gca()

# Arestas com gradiente de opacidade
nx.draw_networkx_edges(G_main, pos,
                      alpha=0.15,
                      width=edge_widths,
                      edge_color='#6B7280',
                      ax=ax)

# Nós
nx.draw_networkx_nodes(G_main, pos,
                      node_size=node_sizes,
                      node_color=node_colors,
                      alpha=0.9,
                      linewidths=2,
                      edgecolors='white',
                      ax=ax)

# Labels apenas para autores principais (reduz poluição visual)
top_by_degree = sorted(degrees_main.items(), key=lambda x: x[1], reverse=True)[:12]
top_by_betweenness = sorted(betweenness_main.items(), key=lambda x: x[1], reverse=True)[:8]
important_authors = set([author for author, _ in top_by_degree] +
                       [author for author, _ in top_by_betweenness])

labels_dict = {author: author for author in important_authors}
nx.draw_networkx_labels(G_main, pos,
                       labels=labels_dict,
                       font_size=8,
                       font_weight='bold',
                       font_color='#1F2937',
                       ax=ax)

# Título
plt.title("Rede de Coautoria - Componente Principal\n"
         f"{len(largest_cc)} autores | {G_main.number_of_edges()} colaborações",
         fontsize=16, fontweight='bold', pad=20)

# Legenda clara e compacta
avg_degree = sum(degrees_main.values()) / len(degrees_main)
legend_elements = [
    mpatches.Patch(facecolor='#E0E7FF', edgecolor='white', linewidth=2,
                  label='Autores com menor centralidade'),
    mpatches.Patch(facecolor='#7C3AED', edgecolor='white', linewidth=2,
                  label='Autores com maior centralidade'),
    mpatches.Patch(facecolor='white', edgecolor='#6B7280', linewidth=1,
                  label='Colaboração (espessura = frequência)'),
]

plt.legend(handles=legend_elements, loc='upper left', fontsize=10,
          frameon=True, fancybox=True, shadow=True)

# Estatísticas no canto inferior
stats_text = (f"Grau médio: {avg_degree:.1f}\n"
             f"Densidade: {nx.density(G_main):.3f}\n"
             f"Diâmetro: {nx.diameter(G_main)}")
plt.figtext(0.02, 0.02, stats_text, fontsize=9,
           bbox=dict(boxstyle="round,pad=0.7", facecolor="#F3F4F6", alpha=0.9))

plt.axis('off')
plt.tight_layout()
plt.show()

# ============================================================
# VISUALIZAÇÃO 2: VISÃO GERAL (SIMPLIFICADA)
# ============================================================
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 8))

# Gráfico 2A: Distribuição de componentes
component_sizes = [len(comp) for comp in components]
comp_categories = {'Isolados (1)': 0, 'Duplas (2)': 0, 'Pequenos (3-4)': 0, 'Médios (5-9)': 0, 'Grandes (≥10)': 0}

for size in component_sizes:
    if size == 1:
        comp_categories['Isolados (1)'] += 1
    elif size == 2:
        comp_categories['Duplas (2)'] += 1
    elif size <= 4:
        comp_categories['Pequenos (3-4)'] += 1
    elif size <= 9:
        comp_categories['Médios (5-9)'] += 1
    else:
        comp_categories['Grandes (≥10)'] += 1

colors_bar = ['#D1D5DB', '#93C5FD', '#FBBF24', '#FB923C', '#EF4444']
bars = ax1.bar(range(len(comp_categories)), list(comp_categories.values()), color=colors_bar, alpha=0.8, edgecolor='black', linewidth=1.5)
ax1.set_xticks(range(len(comp_categories)))
ax1.set_xticklabels(comp_categories.keys(), rotation=45, ha='right')
ax1.set_ylabel('Quantidade de Componentes', fontsize=11, fontweight='bold')
ax1.set_title('Distribuição de Componentes Conectados', fontsize=12, fontweight='bold')
ax1.grid(axis='y', alpha=0.3)

# Adicionar valores nas barras
for bar in bars:
    height = bar.get_height()
    ax1.text(bar.get_x() + bar.get_width()/2., height,
            f'{int(height)}',
            ha='center', va='bottom', fontweight='bold')

# Gráfico 2B: Top 10 colaboradores
top_10 = sorted(degrees.items(), key=lambda x: x[1], reverse=True)[:10]
authors_top, collabs_top = zip(*top_10)
colors_collab = plt.cm.Purples(np.linspace(0.5, 0.9, len(authors_top)))

bars2 = ax2.barh(range(len(authors_top)), collabs_top, color=colors_collab, edgecolor='black', linewidth=1.5)
ax2.set_yticks(range(len(authors_top)))
ax2.set_yticklabels(authors_top, fontsize=10)
ax2.set_xlabel('Número de Colaborações', fontsize=11, fontweight='bold')
ax2.set_title('Top 10 Colaboradores', fontsize=12, fontweight='bold')
ax2.grid(axis='x', alpha=0.3)
ax2.invert_yaxis()

# Adicionar valores nas barras
for i, bar in enumerate(bars2):
    width = bar.get_width()
    ax2.text(width, bar.get_y() + bar.get_height()/2.,
            f'{int(width)}',
            ha='left', va='center', fontweight='bold', fontsize=9)

plt.tight_layout()
plt.show()

# ============================================================
# VISUALIZAÇÃO 3: DISTRIBUIÇÃO DE PESOS
# ============================================================
fig, ax = plt.subplots(figsize=(12, 6))

all_weights = [G_main[u][v].get('weight', 1) for u, v in G_main.edges()]
weight_counts = Counter(all_weights)

weights = sorted(weight_counts.keys())
counts = [weight_counts[w] for w in weights]

ax.bar(weights, counts, color='#8B5CF6', alpha=0.7, edgecolor='black', linewidth=1.5)
ax.set_xlabel('Frequência de Colaboração (nº de artigos conjuntos)', fontsize=11, fontweight='bold')
ax.set_ylabel('Número de Parcerias', fontsize=11, fontweight='bold')
ax.set_title('Distribuição de Frequência de Colaborações', fontsize=12, fontweight='bold')
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

# ============================================================
# RESUMO ESTATÍSTICO NO CONSOLE
# ============================================================
print(f"\n{'='*60}")
print(f"📊 RESUMO ESTATÍSTICO DA REDE")
print(f"{'='*60}")
print(f"Autores totais: {G.number_of_nodes()}")
print(f"Colaborações totais: {G.number_of_edges()}")
print(f"Componentes conectados: {len(components)}")
print(f"Densidade da rede: {nx.density(G):.4f}")
print(f"\nComponente principal: {len(largest_cc)} autores ({len(largest_cc)/G.number_of_nodes()*100:.1f}%)")
print(f"{'='*60}\n")

print(f"🏆 TOP 15 COLABORADORES:")
top_degree = sorted(degrees.items(), key=lambda x: x[1], reverse=True)[:15]
for i, (author, degree) in enumerate(top_degree, 1):
    print(f"{i:2d}. {author:<50} {degree} colaborações")